# Beach Occupancy Dataset — Analysis & Cleaning Pipeline

Automated pipeline to prepare clean datasets for TFT training.
Run periodically when new data is collected.

**Filtering steps:**
1. Solar radiation check → validate camera daylight window
2. Hour filter → keep 8h–20h
3. Stuck camera detection → 48h rolling CV < 0.05
4. Seasonality analysis → auto-flag weak/no-signal cameras
5. Manual cache exclusions → known bad cache sources (fixed list)
6. Min rows filter → drop cameras with < 1000 records
7. Export → clean CSVs for TFT training

**Django cameras are filtered automatically** (no manual exclusion list).
Cache has a fixed manual exclusion list for known bad sources.

In [ ]:
CSV_PATH = 'dataset.csv'
OUT_DIR = 'clean_datasets'
HAS_CACHE = True


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import re
import json

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (13, 5)

key_w = {'om_temperature_2m': 'Temp (°C)', 'om_cloud_cover': 'Cloud (%)',
         'om_wind_speed_10m': 'Wind (m/s)', 'om_shortwave_radiation': 'Solar (W/m²)',
         'om_relative_humidity_2m': 'Humidity (%)', 'om_precipitation': 'Precip (mm)'}
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
colors_dow = ['#3498db'] * 5 + ['#e74c3c'] * 2


In [ ]:
BASE_RC = {
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Liberation Serif', 'Nimbus Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 20, 'axes.titlesize': 22, 'axes.labelsize': 20,
    'xtick.labelsize': 17, 'ytick.labelsize': 17,
    'legend.fontsize': 17, 'figure.titlesize': 24,
}
plt.rcParams.update(BASE_RC)

MID_RC = {
    'font.size': 13, 'axes.titlesize': 15, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'legend.fontsize': 11, 'figure.titlesize': 17,
}


In [ ]:
raw = pd.read_csv(CSV_PATH, parse_dates=['ds'])

om_cols = [c for c in raw.columns if c.startswith('om_')]
weather_cols = om_cols

print(f"Shape:      {raw.shape}")
print(f"Date range: {raw['ds'].min()} → {raw['ds'].max()}")
print(f"Beaches:    {raw['unique_id'].nunique()}")
print(f"Sources:    {raw['source'].value_counts().to_dict()}")
print(f"Columns:    {len(om_cols)} Open-Meteo weather")
raw.head(3)

In [ ]:
# Thesis figure: solar radiation vs crowd count by hour (8:00-20:00 daytime frame).
# Grey band = excluded hours (before 08:00 and from 21:00). Exported at IEEE \textwidth (serif 8pt).
import matplotlib.pyplot as plt
from pathlib import Path
_RC = {'font.family':'serif','font.serif':['Times New Roman','DejaVu Serif'],'mathtext.fontset':'stix',
       'font.size':8,'axes.titlesize':8,'axes.labelsize':8,'xtick.labelsize':7,'ytick.labelsize':7,
       'legend.fontsize':7,'axes.linewidth':0.6,'lines.linewidth':1.2}
_FIG = Path.home()/'Desktop/master_activities/TFM/beach_counting/TFM_organized/MUSI___TFM_Template/figures'
_uv = 'om_shortwave_radiation'
_summer = raw[raw['month'].isin([6,7,8])]; _winter = raw[raw['month'].isin([12,1,2])]
with plt.rc_context(_RC):
    fig, axes = plt.subplots(2, 2, figsize=(7.139, 4.6), sharex=True)
    for i,(sub,lab,col) in enumerate([(_summer,'Summer (Jun-Aug)','#e67e22'),(_winter,'Winter (Dec-Feb)','#2980b9')]):
        axes[0,i].scatter(sub['hour'],sub[_uv],s=1,alpha=0.15,color=col,rasterized=True)
        hm=sub.groupby('hour')[_uv].mean(); axes[0,i].plot(hm.index,hm.values,'k-o',ms=3,lw=1.2,label='mean')
        axes[0,i].axvspan(-0.5,7.5,color='#2c3e50',alpha=0.1); axes[0,i].axvspan(20.5,23.5,color='#2c3e50',alpha=0.1)
        axes[0,i].set_title(lab); axes[0,i].set_ylabel(r'Shortwave (W/m$^2$)'); axes[0,i].legend()
        axes[1,i].scatter(sub['hour'],sub['y'],s=1,alpha=0.15,color=col,rasterized=True)
        h2=sub.groupby('hour')['y'].mean(); axes[1,i].plot(h2.index,h2.values,'k-o',ms=3,lw=1.2,label='mean')
        axes[1,i].axvspan(-0.5,7.5,color='#2c3e50',alpha=0.1); axes[1,i].axvspan(20.5,23.5,color='#2c3e50',alpha=0.1)
        axes[1,i].set_xlabel('Hour'); axes[1,i].set_ylabel('Crowd count'); axes[1,i].legend()
    fig.tight_layout(); fig.savefig(_FIG/'fig_solar_vs_crowd_by_hour.png', dpi=300, bbox_inches='tight'); plt.show()


In [ ]:
plt.rcParams.update(MID_RC)
uv_om = 'om_shortwave_radiation'
threshold_om = 50   # W/m² — enough light for camera reliability

summer = raw[raw['month'].isin([6, 7, 8])]
winter = raw[raw['month'].isin([12, 1, 2])]

print(f"{'':30s} {'first':>8s} {'last':>8s} {'window':>8s}  {'first':>8s} {'last':>8s} {'window':>8s}")
print('-' * 95)

for label, subset in [('Summer (Jun-Aug)', summer), ('Winter (Dec-Feb)', winter)]:
    # Open-Meteo
    om_hourly = subset.groupby('hour')[uv_om].mean()
    om_good = om_hourly[om_hourly > threshold_om]
    om_first = om_good.index.min() if len(om_good) > 0 else None
    om_last = om_good.index.max() if len(om_good) > 0 else None


    om_win = f"{om_last - om_first + 1}h" if om_first is not None else '-'

    print(f"  {label:28s} {om_first or '-':>8}h {om_last or '-':>7}h {om_win:>8s}  ")

# Visual
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for i, (subset, label, color) in enumerate([
    (summer, 'Summer', '#e67e22'), (winter, 'Winter', '#2980b9')
]):
    om_h = subset.groupby('hour')[uv_om].mean()
    axes[i].bar(om_h.index, om_h.values, color=color, alpha=0.7, label='OM shortwave')

    ax2 = axes[i].twinx()

    axes[i].axhline(threshold_om, color='red', ls='--', lw=1, alpha=0.7, label=f'OM threshold ({threshold_om} W/m²)')
    om_good = om_h[om_h > threshold_om]
    if len(om_good) > 0:
        axes[i].axvspan(om_good.index.min() - 0.5, om_good.index.max() + 0.5,
                        color='#f1c40f', alpha=0.15, label=f'Usable window: {om_good.index.min()}h–{om_good.index.max()}h')

    axes[i].set_title(f'{label} — Solar Radiation by Hour')
    axes[i].set_xlabel('Hour')
    axes[i].set_ylabel('Shortwave (W/m²)')
    axes[i].set_xticks(range(0, 24))
    axes[i].legend(loc='upper left', fontsize=7)
    ax2.legend(loc='upper right', fontsize=7)

plt.tight_layout()
plt.show()

print("\n→ Suggested safe filtering windows:")
for label, subset in [('Summer', summer), ('Winter', winter)]:
    om_h = subset.groupby('hour')[uv_om].mean()
    good = om_h[om_h > threshold_om]
    if len(good) > 0:
        print(f"  {label}: hour >= {good.index.min()} and hour <= {good.index.max()}")
plt.rcParams.update(BASE_RC)


---
# Part 0 — Night & Low-Light Filtering

Beach cameras cannot count people accurately without daylight.
We apply a **fixed hour filter (8:00–20:00)** based on summer daylight hours in the Balearic Islands,
since summer is the primary period of interest for beach occupancy prediction.

This keeps the approach simple and reproducible — no dependency on weather API availability.


In [ ]:
plt.rcParams.update(MID_RC)
HOUR_MIN, HOUR_MAX = 8, 20

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# Records per hour
hourly_counts = raw.groupby('hour').size()
colors = ['#2ecc71' if HOUR_MIN <= h <= HOUR_MAX else '#e74c3c' for h in hourly_counts.index]
hourly_counts.plot.bar(ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Records per Hour (green = kept)')
axes[0].set_ylabel('Count')

# Mean y per hour
hourly_y = raw.groupby('hour')['y'].mean()
colors_y = ['#2ecc71' if HOUR_MIN <= h <= HOUR_MAX else '#e74c3c' for h in hourly_y.index]
hourly_y.plot.bar(ax=axes[1], color=colors_y, edgecolor='white')
axes[1].set_title('Mean Crowd Count per Hour')
axes[1].set_ylabel('Mean y')

# Distribution: kept vs removed
kept = raw[(raw['hour'] >= HOUR_MIN) & (raw['hour'] <= HOUR_MAX)]['y']
removed = raw[(raw['hour'] < HOUR_MIN) | (raw['hour'] > HOUR_MAX)]['y']
axes[2].hist(kept, bins=50, alpha=0.6, label=f'8-20h (n={len(kept):,})', density=True, color='#2ecc71')
axes[2].hist(removed, bins=50, alpha=0.6, label=f'Outside (n={len(removed):,})', density=True, color='#e74c3c')
axes[2].set_title('y Distribution: Kept vs Removed')
axes[2].set_xlabel('y')
axes[2].set_xlim(0, raw['y'].quantile(0.98))
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'Kept (8-20h):    {len(kept):,} records ({len(kept)/len(raw)*100:.1f}%)  mean y={kept.mean():.1f}')
print(f'Removed:         {len(removed):,} records ({len(removed)/len(raw)*100:.1f}%)  mean y={removed.mean():.1f}')

plt.rcParams.update(BASE_RC)


In [ ]:
HOUR_MIN, HOUR_MAX = 8, 20

time_mask = (raw['hour'] >= HOUR_MIN) & (raw['hour'] <= HOUR_MAX)
df = raw[time_mask].copy().reset_index(drop=True)

print(f'Filter: hours {HOUR_MIN}–{HOUR_MAX}')
print(f'Filtered: {len(raw):,} → {len(df):,} records (removed {(~time_mask).sum():,})')
print(f'Remaining hours: {sorted(df["hour"].unique())}')

for label, months in [('Summer', [6,7,8]), ('Winter', [12,1,2])]:
    sub = df[df['month'].isin(months)]
    if len(sub) > 0:
        print(f'  {label}: {len(sub):,} records')

cache = df[df['source'] == 'cache'].copy() if HAS_CACHE else pd.DataFrame()
django = df[df['source'] == 'django'].copy()

if HAS_CACHE:
    print(f'\nCache (2022-2023):  {len(cache):,} records, {cache["unique_id"].nunique()} beaches')
print(f'Django (2025-2026): {len(django):,} records, {django["unique_id"].nunique()} beaches')
if not HAS_CACHE:
    print('\nRunning in django-only mode (no cache data)')

In [ ]:
df = pd.concat([cache, django], ignore_index=True) if HAS_CACHE else django.copy()

if HAS_CACHE:
    print(f'Cache:  {len(cache):,} records')
print(f'Django: {len(django):,} records')
print(f'Total:  {len(df):,} records')

---
# Part 1 — Cache Period (2022-2023)

Historical predictions from the Bayesian VGG-19 model with Open-Meteo weather.

## 1.1 Overview

In [ ]:
if not HAS_CACHE:
    print("Skipping cache analysis (django-only mode)")
else:
    print(f"Shape:      {cache.shape}")
    print(f"Date range: {cache['ds'].min()} → {cache['ds'].max()}")
    print(f"Beaches:    {cache['unique_id'].nunique()}")
    print(f"Open-Meteo: {cache[om_cols].notna().all(axis=1).mean()*100:.1f}% complete")
    
    cache_summary = cache.groupby('unique_id').agg(
        n=('y', 'size'), mean=('y', 'mean'), std=('y', 'std'),
        min=('y', 'min'), max=('y', 'max'),
        first=('ds', 'min'), last=('ds', 'max'),
    ).round(1)
    cache_summary['cv'] = (cache_summary['std'] / cache_summary['mean']).round(3)
    cache_summary.sort_values('n', ascending=False)

## 1.2 Crowd Count Distribution

In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    cache['y'].hist(bins=60, ax=axes[0], color='#2ecc71', edgecolor='white', alpha=0.8)
    axes[0].axvline(cache['y'].median(), color='red', ls='--', label=f"median={cache['y'].median():.0f}")
    axes[0].set_title('Cache — Distribution of y')
    axes[0].set_xlabel('Crowd count')
    axes[0].legend()
    
    np.log1p(cache['y']).hist(bins=60, ax=axes[1], color='#9b59b6', edgecolor='white', alpha=0.8)
    axes[1].set_title('Cache — log(1+y)')
    
    order_c = cache.groupby('unique_id')['y'].median().sort_values().index
    sns.boxplot(data=cache, y='unique_id', x='y', order=order_c, ax=axes[2],
                fliersize=1, linewidth=0.5)
    axes[2].set_title('Cache — y by Beach')
    axes[2].set_ylabel('')
    
    plt.tight_layout()
    plt.show()
    
    print(cache['y'].describe().round(2))
    print(f"Skewness: {cache['y'].skew():.2f}")
plt.rcParams.update(BASE_RC)


## 1.3 Time Series (scatter)

In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE:
    c_beaches = sorted(cache['unique_id'].unique())
    n_c = len(c_beaches)
    cols = 2
    rows_c = (n_c + 1) // cols
    
    fig, axes = plt.subplots(rows_c, cols, figsize=(16, 3 * rows_c), sharex=False, squeeze=False)
    axes = np.array(axes).flatten()
    
    for i, beach in enumerate(c_beaches):
        bdf = cache[cache['unique_id'] == beach].sort_values('ds')
        cv = bdf['y'].std() / bdf['y'].mean() if bdf['y'].mean() > 0 else 0
        axes[i].scatter(bdf['ds'], bdf['y'], s=1, alpha=0.3, rasterized=True)
        axes[i].set_title(f"{beach}  (n={len(bdf):,}, CV={cv:.2f})")
        axes[i].set_ylabel('y')
        axes[i].tick_params(axis='x', rotation=30, labelsize=7)
    
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    fig.suptitle('Cache (2022-2023) — Raw Crowd Count Scatter', y=1.01)
    plt.tight_layout()
    plt.show()
plt.rcParams.update(BASE_RC)


### Seasonality Slope Analysis — Camera Validation

**Theory:** A legitimate beach camera should show a strong seasonal signal:
summer occupancy significantly higher than winter.

If a camera shows **no significant seasonal slope** (summer ≈ winter), it likely means:
- The model is counting **background objects** rather than people
- The camera is **malfunctioning** or has a fixed noise floor
- The camera is **not pointing at a beach**

We measure this with:
- **Seasonal ratio** = `summer_mean / winter_mean` — a healthy beach should be >1.5×
- **Effect size** (Cohen's d) — robust to scale differences across beaches

**Fallback for summer-only datasets:** When a beach has data only from May–September
(no winter comparison possible), we compare **peak** (Jul–Aug) vs **shoulder** (May, Jun, Sep, Oct).
A beach with a clear bell curve (e.g., 50 → 120 → 100) has seasonal signal even without winter data.


In [ ]:
# helper to get identifiers for a beach
def get_beach_ids(beach):
    bdf = raw[raw['unique_id'] == beach]
    wid = bdf['webcam_id'].replace('', np.nan).dropna().unique()
    slug = bdf['slug'].replace('', np.nan).dropna().unique()
    folder = bdf['beach_folder'].replace('', np.nan).dropna().unique()
    parts = []
    if len(wid) > 0:
        parts.append(f"webcam_id={','.join(str(w) for w in wid)}")
    if len(slug) > 0:
        parts.append(f"slug={slug[0]}")
    if len(folder) > 0:
        parts.append(f"folder={','.join(folder)}")
    return ' | '.join(parts) if parts else '-'

In [ ]:
plt.rcParams.update({'font.size': 14, 'axes.titlesize': 18, 'axes.labelsize': 16, 'ytick.labelsize': 10, 'xtick.labelsize': 13, 'legend.fontsize': 11})
# Seasonality slope analysis — SEPARATE per period
summer_months = [6, 7, 8]
winter_months = [12, 1, 2]
warm_months = [5, 6, 7, 8, 9]
cold_months = [11, 12, 1, 2, 3]

# --- Step 1: Remove stuck/frozen camera periods (rolling CV) ---
CV_WINDOW = 48
CV_THRESHOLD = 0.05

def remove_stuck(subset):
    clean_parts = []
    stuck_stats = []
    for (beach, src), grp in subset.groupby(['unique_id', 'source']):
        grp = grp.sort_values('ds').copy()
        if len(grp) < CV_WINDOW:
            clean_parts.append(grp)
            continue

        roll_mean = grp['y'].rolling(CV_WINDOW, center=True, min_periods=12).mean()
        roll_std = grp['y'].rolling(CV_WINDOW, center=True, min_periods=12).std()
        roll_cv = (roll_std / roll_mean).where(roll_mean > 0, np.nan)

        stuck = roll_cv < CV_THRESHOLD
        stuck = stuck.fillna(False)

        n_stuck = stuck.sum()
        if n_stuck > 0:
            stuck_stats.append({
                'beach': beach, 'source': src, 'total': len(grp),
                'stuck': int(n_stuck), 'pct': round(n_stuck / len(grp) * 100, 1),
                'example_y': round(grp.loc[stuck, 'y'].median(), 1),
            })

        clean_parts.append(grp[~stuck])

    result = pd.concat(clean_parts, ignore_index=True) if clean_parts else pd.DataFrame()
    stats_df = pd.DataFrame(stuck_stats).sort_values('pct', ascending=False) if stuck_stats else pd.DataFrame()
    return result, stats_df

df_clean, stuck_stats = remove_stuck(df)
n_removed = len(df) - len(df_clean)
print(f"Stuck camera filter (48h rolling CV < {CV_THRESHOLD}): "
      f"{len(df):,} -> {len(df_clean):,} (removed {n_removed:,})")

if len(stuck_stats) > 0:
    print(f"\nAffected cameras ({len(stuck_stats)}):")
    print(stuck_stats.to_string(index=False))
else:
    print("No stuck periods detected.")

cache_all = df_clean[df_clean['source'].isin(['cache', 'both'])].copy()
django_all = df_clean[df_clean['source'].isin(['django', 'both'])].copy()

# --- Helper: seasonality analysis ---
def analyse_seasonality(subset):
    peak_months = [7, 8]
    shoulder_months = [5, 6, 9, 10]

    beaches = sorted(subset['unique_id'].unique())
    results = []
    for beach in beaches:
        bdf = subset[subset['unique_id'] == beach]

        s = bdf[bdf['month'].isin(summer_months)]['y']
        w = bdf[bdf['month'].isin(winter_months)]['y']
        if len(s) < 10:
            s = bdf[bdf['month'].isin(warm_months)]['y']
        if len(w) < 10:
            w = bdf[bdf['month'].isin(cold_months)]['y']

        if len(s) >= 10 and len(w) < 10:
            peak = bdf[bdf['month'].isin(peak_months)]['y']
            shoulder = bdf[bdf['month'].isin(shoulder_months)]['y']

            if len(peak) >= 10 and len(shoulder) >= 10:
                ratio = peak.mean() / shoulder.mean() if shoulder.mean() > 0 else np.nan
                pooled_std = np.sqrt((peak.std()**2 + shoulder.std()**2) / 2)
                cohens_d = (peak.mean() - shoulder.mean()) / pooled_std if pooled_std > 0 else 0
                amplitude = peak.mean() - shoulder.mean()

                if ratio > 1.5 or cohens_d > 0.5:
                    cls = '🟢 Strong (peak vs shoulder)'
                elif ratio > 1.2:
                    cls = '🟡 Weak (peak vs shoulder)'
                else:
                    cls = '🔴 No signal (peak vs shoulder)'

                results.append({
                    'beach': beach, 'n': len(bdf),
                    'summer_mean': round(peak.mean(), 1),
                    'winter_mean': round(shoulder.mean(), 1),
                    'ratio': round(ratio, 2) if not np.isnan(ratio) else np.nan,
                    'cohens_d': round(cohens_d, 2), 'amplitude': round(amplitude, 1),
                    'classification': cls,
                    'summer_n': len(peak), 'winter_n': len(shoulder),
                })
                continue

        if len(s) < 10 or len(w) < 10:
            results.append({'beach': beach, 'n': len(bdf),
                            'summer_mean': s.mean() if len(s) > 0 else np.nan,
                            'winter_mean': np.nan, 'ratio': np.nan, 'cohens_d': np.nan,
                            'amplitude': np.nan, 'classification': '⚪ Insufficient data',
                            'summer_n': len(s), 'winter_n': len(w)})
            continue

        ratio = s.mean() / w.mean() if w.mean() > 0 else np.nan
        pooled_std = np.sqrt((s.std()**2 + w.std()**2) / 2)
        cohens_d = (s.mean() - w.mean()) / pooled_std if pooled_std > 0 else 0
        amplitude = s.mean() - w.mean()

        if ratio > 2 or cohens_d > 0.5:
            cls = '🟢 Strong'
        elif ratio > 1.3:
            cls = '🟡 Weak'
        else:
            cls = '🔴 No signal'

        results.append({
            'beach': beach, 'n': len(bdf),
            'summer_mean': round(s.mean(), 1), 'winter_mean': round(w.mean(), 1),
            'ratio': round(ratio, 2) if not np.isnan(ratio) else np.nan,
            'cohens_d': round(cohens_d, 2), 'amplitude': round(amplitude, 1),
            'classification': cls, 'summer_n': len(s), 'winter_n': len(w),
        })

    result_df = pd.DataFrame(results)
    return result_df.sort_values('ratio', ascending=True, na_position='first') if not result_df.empty else result_df

CLS_COLORS = {'Strong': '#2ecc71', 'Weak': '#f39c12', 'No signal': '#e74c3c', 'Insufficient': '#95a5a6'}

def get_cls_color(cls_str):
    for key, color in CLS_COLORS.items():
        if key in cls_str:
            return color
    return '#95a5a6'

# --- Helper: plot summary 2x2 ---
def plot_summary(season_df, subset, title):
    if season_df.empty or season_df.dropna(subset=['ratio']).empty:
        print(f'{title}: no data to plot')
        return
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), squeeze=False)

    valid = season_df.dropna(subset=['ratio'])
    colors = [get_cls_color(c) for c in valid['classification']]
    if not valid.empty:
        axes[0, 0].barh(valid['beach'], valid['ratio'], color=colors, alpha=0.8)
    axes[0, 0].axvline(1.3, color='orange', ls='--', lw=1, label='Weak (1.3x)')
    axes[0, 0].axvline(2.0, color='green', ls='--', lw=1, label='Strong (2.0x)')
    axes[0, 0].axvline(1.0, color='red', ls='-', lw=1, alpha=0.5, label='1x (no seasonality)')
    axes[0, 0].set_title('Seasonal Ratio')
    axes[0, 0].set_xlabel('Ratio')
    axes[0, 0].legend(fontsize=7)

    valid2 = season_df.dropna(subset=['summer_mean', 'winter_mean'])
    colors2 = [get_cls_color(c) for c in valid2['classification']]
    axes[0, 1].scatter(valid2['winter_mean'], valid2['summer_mean'], c=colors2, s=60,
                       edgecolors='black', linewidths=0.5, alpha=0.8)
    for _, row in valid2.iterrows():
        axes[0, 1].annotate(row['beach'], (row['winter_mean'], row['summer_mean']),
                            fontsize=5, xytext=(3, 3), textcoords='offset points')
    lim = max(valid2[['summer_mean', 'winter_mean']].max().max() * 1.1, 1)
    axes[0, 1].plot([0, lim], [0, lim], 'r--', alpha=0.4, label='1:1')
    axes[0, 1].plot([0, lim], [0, lim * 2], 'g--', alpha=0.3, label='2:1')
    axes[0, 1].set_xlabel('Winter mean y')
    axes[0, 1].set_ylabel('Summer mean y')
    axes[0, 1].set_title('Summer vs Winter Mean')
    axes[0, 1].legend(fontsize=7)

    valid3 = season_df.dropna(subset=['cohens_d']).sort_values('cohens_d')
    colors3 = [get_cls_color(c) for c in valid3['classification']]
    if not valid3.empty:
        axes[1, 0].barh(valid3['beach'], valid3['cohens_d'], color=colors3, alpha=0.8)
    axes[1, 0].axvline(0.2, color='red', ls='--', lw=1, label='Small (0.2)')
    axes[1, 0].axvline(0.5, color='orange', ls='--', lw=1, label='Medium (0.5)')
    axes[1, 0].axvline(0.8, color='green', ls='--', lw=1, label='Large (0.8)')
    axes[1, 0].set_title("Cohen's d")
    axes[1, 0].set_xlabel("Cohen's d")
    axes[1, 0].legend(fontsize=7)

    suspects = season_df[season_df['classification'].str.contains('\U0001f534')]
    if len(suspects) > 0:
        for _, row in suspects.head(6).iterrows():
            bdf = subset[subset['unique_id'] == row['beach']]
            monthly = bdf.groupby('month')['y'].mean()
            axes[1, 1].plot(monthly.index, monthly.values, 'o-', ms=4, label=row['beach'])
        axes[1, 1].set_title('Monthly Profile - Suspect Cameras')
    else:
        strong = season_df[season_df['classification'].str.contains('\U0001f7e2')].tail(4)
        for _, row in strong.iterrows():
            bdf = subset[subset['unique_id'] == row['beach']]
            monthly = bdf.groupby('month')['y'].mean()
            axes[1, 1].plot(monthly.index, monthly.values, 'o-', ms=4, label=row['beach'])
        axes[1, 1].set_title('Monthly Profile - Strongest Signals')
    axes[1, 1].set_xlabel('Month')
    axes[1, 1].set_ylabel('Mean y')
    axes[1, 1].set_xticks(range(1, 13))
    axes[1, 1].legend(fontsize=6, ncol=2)

    fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()

# --- Helper: per-beach scatter + trend (colored by classification) ---
def plot_scatter(subset, season_df, title):
    beaches = sorted(subset['unique_id'].unique())
    n_b = len(beaches)
    if n_b == 0:
        return
    cols_p = 3
    rows_b = (n_b + cols_p - 1) // cols_p

    fig, axes = plt.subplots(rows_b, cols_p, figsize=(17, 3.5 * rows_b), sharex=False, squeeze=False)
    axes = np.array(axes).flatten()

    for i, beach in enumerate(beaches):
        bdf = subset[subset['unique_id'] == beach].sort_values('ds')
        row_info = season_df[season_df['beach'] == beach]
        cls = row_info['classification'].values[0] if len(row_info) > 0 else '?'
        ratio = row_info['ratio'].values[0] if len(row_info) > 0 else np.nan
        color = get_cls_color(cls)

        axes[i].scatter(bdf['ds'], bdf['y'], s=1, alpha=0.2, color=color, rasterized=True)

        daily = bdf.set_index('ds')['y'].resample('D').mean().dropna()
        if len(daily) > 30:
            trend = daily.rolling(30, center=True, min_periods=10).mean()
            axes[i].plot(trend.index, trend.values, color='black', lw=1.5, label='30d trend')

        is_shoulder = 'shoulder' in str(row_info['classification'].values[0]) if len(row_info) > 0 else False
        if is_shoulder:
            s_mean = bdf[bdf['month'].isin([7, 8])]['y'].mean()
            w_mean = bdf[bdf['month'].isin([5, 6, 9, 10])]['y'].mean()
            s_label, w_label = 'peak', 'shoulder'
        else:
            s_mean = bdf[bdf['month'].isin(summer_months)]['y'].mean()
            w_mean = bdf[bdf['month'].isin(winter_months)]['y'].mean()
            s_label, w_label = 'summer', 'winter'
        if not np.isnan(s_mean):
            axes[i].axhline(s_mean, color='#e74c3c', ls='--', lw=1, alpha=0.6, label=f'{s_label}={s_mean:.0f}')
        if not np.isnan(w_mean):
            axes[i].axhline(w_mean, color='#2980b9', ls='--', lw=1, alpha=0.6, label=f'{w_label}={w_mean:.0f}')

        ratio_str = f'{ratio:.1f}x' if not np.isnan(ratio) else 'N/A'
        axes[i].set_title(f"{beach}\n{cls} | ratio={ratio_str}", pad=3)
        axes[i].set_ylabel('y')
        axes[i].tick_params(axis='x', rotation=30, labelsize=6)
        axes[i].tick_params(axis='y', labelsize=7)
        axes[i].legend(fontsize=5, loc='upper right', ncol=2)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

# Defaults
season_cache_df = pd.DataFrame()
suspect_list_c = pd.DataFrame()
insuff_list_c = pd.DataFrame()

# ============================
# CACHE (2022-2023)
# ============================
if HAS_CACHE and len(cache_all) > 0:
    season_cache_df = analyse_seasonality(cache_all)
    if not season_cache_df.empty:
        plot_summary(season_cache_df, cache_all, 'Cache (2022-2023) - Seasonality Slope Analysis')
        plot_scatter(cache_all, season_cache_df, 'Cache (2022-2023) - Per-Beach Scatter + 30d Trend')
        print('CACHE - Classification:')
        print(season_cache_df['classification'].value_counts().to_string())
        suspect_list_c = season_cache_df[season_cache_df['classification'].str.contains('No signal|Weak', na=False)]
        insuff_list_c = season_cache_df[season_cache_df['classification'].str.contains('Insufficient', na=False)]
        if len(suspect_list_c) > 0:
            print(f'\n{len(suspect_list_c)} cameras with NO seasonal signal:')
            for _, row in suspect_list_c.iterrows():
                print(f"  {row['beach']:35s} ratio={row['ratio']}  cohen_d={row['cohens_d']}")
        if len(insuff_list_c) > 0:
            print(f'\n{len(insuff_list_c)} cameras with INSUFFICIENT data:')
            for _, row in insuff_list_c.iterrows():
                print(f"  {row['beach']:35s} summer_n={row['summer_n']}  winter_n={row['winter_n']}")
    else:
        print('Cache: no cameras with summer+winter data')

# ============================
# DJANGO (2025-2026)
# ============================
season_django_df = analyse_seasonality(django_all)
suspect_list_d = pd.DataFrame()
insuff_list_d = pd.DataFrame()

if not season_django_df.empty:
    plot_summary(season_django_df, django_all, 'Django (2025-2026) - Seasonality Slope Analysis')
    plot_scatter(django_all, season_django_df, 'Django (2025-2026) - Per-Beach Scatter + 30d Trend')
    print('DJANGO - Classification:')
    print(season_django_df['classification'].value_counts().to_string())
    suspect_list_d = season_django_df[season_django_df['classification'].str.contains('No signal|Weak', na=False)]
    insuff_list_d = season_django_df[season_django_df['classification'].str.contains('Insufficient', na=False)]
    if len(suspect_list_d) > 0:
        print(f'\n{len(suspect_list_d)} cameras with NO seasonal signal:')
        for _, row in suspect_list_d.iterrows():
            print(f"  {row['beach']:35s} ratio={row['ratio']}  cohen_d={row['cohens_d']}")
    if len(insuff_list_d) > 0:
        print(f'\n{len(insuff_list_d)} cameras with INSUFFICIENT data:')
        for _, row in insuff_list_d.iterrows():
            print(f"  {row['beach']:35s} summer_n={row['summer_n']}  winter_n={row['winter_n']}")
else:
    print('Django: no cameras with summer+winter data')

plt.rcParams.update(BASE_RC)


In [ ]:
# Thesis figure: camera validation by seasonal ratio (IEEE 8pt, savefig).
# Uses season_django_df from the seasonality cell above; keeps only the summer-vs-winter
# classified series (drops the peak-vs-shoulder fallback and insufficient-data cameras),
# matching the summer/winter definition used in the thesis text.
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
_RC = {'font.family':'serif','font.serif':['Times New Roman','DejaVu Serif'],'mathtext.fontset':'stix',
       'font.size':8,'axes.labelsize':8,'xtick.labelsize':7,'ytick.labelsize':7,'legend.fontsize':7,'axes.linewidth':0.6}
_FIG = Path.home()/'Desktop/master_activities/TFM/beach_counting/TFM_organized/MUSI___TFM_Template/figures'
_pal = {'Strong':'#2ecc71','Weak':'#f39c12','No signal':'#e74c3c'}
def _cc(c): return 'Strong' if 'Strong' in c else ('Weak' if 'Weak' in c else 'No signal')
_sw = season_django_df[~season_django_df['classification'].str.contains('peak vs shoulder|Insufficient', na=False)]
_s = _sw.dropna(subset=['summer_mean','winter_mean','ratio']).sort_values('ratio').reset_index(drop=True)
_XC = 12.0
with plt.rc_context(_RC):
    fig, ax = plt.subplots(figsize=(7.139, max(4.8, 0.155*len(_s))))
    ax.barh(range(len(_s)), _s['ratio'].clip(upper=_XC), color=[_pal[_cc(c)] for c in _s['classification']], edgecolor='black', linewidth=0.3, zorder=3)
    for i, r in enumerate(_s['ratio']):
        if r > _XC: ax.annotate(f'{r:.0f}x', (_XC, i), fontsize=6, va='center', xytext=(2,0), textcoords='offset points')
    ax.set_yticks(range(len(_s))); ax.set_yticklabels(_s['beach'], fontsize=8); ax.set_ylim(-0.6, len(_s)-0.4)
    ax.axvline(1.0, color='0.45', ls='-', lw=0.7, alpha=0.6, zorder=2)
    ax.axvline(1.3, color='#f39c12', ls='--', lw=1.1, zorder=2)
    ax.axvline(2.0, color='#2ecc71', ls='--', lw=1.1, zorder=2)
    ax.set_xlim(0, _XC+0.6); ax.set_xlabel(r'Summer / winter mean ratio (bars capped at 12$\times$)')
    _h = [Patch(facecolor=_pal[c], edgecolor='black', label=l) for c,l in [('Strong','Strong (kept)'),('Weak','Weak (flagged)'),('No signal','No signal (excluded)')]]
    _h += [plt.Line2D([],[],color='0.45',ls='-',label=r'1$\times$ (no seasonality)'), plt.Line2D([],[],color='#2ecc71',ls='--',label=r'2$\times$ (strong)'), plt.Line2D([],[],color='#f39c12',ls='--',label=r'1.3$\times$ (weak)')]
    ax.legend(handles=_h, fontsize=8, loc='lower right', framealpha=0.95)
    fig.savefig(_FIG/'fig_camval_scatter.png', dpi=300, bbox_inches='tight'); plt.show()


In [ ]:
# Thesis figure: per-beach Cohen's d (effect-size half of the seasonality test), IEEE 8pt, savefig.
# Same summer-vs-winter series as the ratio figure; green bars left of d=0.5 are kept by the ratio (OR rule).
import matplotlib.pyplot as plt
from pathlib import Path
_RC = {'font.family':'serif','font.serif':['Times New Roman','DejaVu Serif'],'mathtext.fontset':'stix',
       'font.size':8,'axes.labelsize':8,'xtick.labelsize':7,'ytick.labelsize':7,'legend.fontsize':7,'axes.linewidth':0.6}
_FIG = Path.home()/'Desktop/master_activities/TFM/beach_counting/TFM_organized/MUSI___TFM_Template/figures'
_pal = {'Strong':'#2ecc71','Weak':'#f39c12','No signal':'#e74c3c'}
def _cc(c): return 'Strong' if 'Strong' in c else ('Weak' if 'Weak' in c else 'No signal')
_sw = season_django_df[~season_django_df['classification'].str.contains('peak vs shoulder|Insufficient', na=False)]
_d = _sw.dropna(subset=['cohens_d']).sort_values('cohens_d').reset_index(drop=True)
_XC = 3.0
with plt.rc_context(_RC):
    fig, ax = plt.subplots(figsize=(7.139, max(4.8, 0.155*len(_d))))
    ax.barh(range(len(_d)), _d['cohens_d'].clip(upper=_XC), color=[_pal[_cc(c)] for c in _d['classification']], edgecolor='black', lw=0.3, zorder=3)
    for i, dv in enumerate(_d['cohens_d']):
        if dv > _XC: ax.annotate(f'{dv:.1f}', (_XC, i), fontsize=6, va='center', xytext=(2,0), textcoords='offset points')
    ax.set_yticks(range(len(_d))); ax.set_yticklabels(_d['beach'], fontsize=8)
    ax.set_xlim(min(-0.3, _d['cohens_d'].min()-0.1), _XC+0.5)
    ax.axvline(0.0, color='0.45', lw=0.7, alpha=0.6, zorder=2)
    ax.axvline(0.2, color='0.6', ls=':', lw=1, zorder=2)
    ax.axvline(0.5, color='#2ecc71', ls='--', lw=1.2, zorder=2, label='$d=0.5$ (medium, kept)')
    ax.axvline(0.8, color='0.6', ls=':', lw=1, zorder=2, label='$d=0.2/0.8$ (small/large)')
    ax.set_xlabel(r"Cohen's $d$ (summer vs winter, standardised by pooled SD; capped at 3)")
    ax.legend(fontsize=8, loc='lower right', framealpha=0.95)
    fig.savefig(_FIG/'fig_cohen_scatter.png', dpi=300, bbox_inches='tight'); plt.show()


### Visual Inspection of Flagged Datasets

Before removing datasets, inspect sample images from each flagged beach.
This helps distinguish:
- **True problems** (camera counting chairs, pointing at harbour) → exclude
- **Partial problems** (bad start, good later; camera repositioned mid-season) → may keep with `FORCE_KEEP`

Images from:
- **Cache (2022):** local files in `BeachCamDataset_NoVideos/{beach_folder}/`
- **Django (2025):** `https://ocupacioplatges.uib.eu/media/img/predictions/{slug}_{webcam_id}_{timestamp}.jpg`

In [ ]:
plt.rcParams.update(MID_RC)
from pathlib import Path
from urllib.request import urlopen, Request
from urllib.error import URLError, HTTPError
from PIL import Image, ImageFile
from io import BytesIO

ImageFile.LOAD_TRUNCATED_IMAGES = True

CACHE_ROOT = Path('/Users/guilhermemossibento/Desktop/master_activities/TFM/BeachCamDataset_NoVideos')
DJANGO_BASE = 'https://ocupacioplatges.uib.eu/media'
N_SAMPLES = 6

def get_cache_images(beach, subset, n=N_SAMPLES):
    bdf = raw[raw['unique_id'] == beach]
    folders = bdf['beach_folder'].replace('', np.nan).dropna().unique()
    if len(folders) == 0:
        return [], []
    folder = CACHE_ROOT / folders[0]
    if not folder.exists():
        return [], []
    imgs = sorted(folder.glob('*.jpg')) + sorted(folder.glob('*.png'))
    if len(imgs) == 0:
        return [], []
    indices = np.linspace(0, len(imgs) - 1, n, dtype=int)
    selected = [imgs[i] for i in indices]
    labels = [img.stem for img in selected]
    return selected, labels

def get_django_images(beach, subset, n=N_SAMPLES):
    bdf = subset[subset['unique_id'] == beach].sort_values('ds')
    if len(bdf) == 0:
        return [], []
    
    path_col = None
    for col in ['prediction_path', 'image_path']:
        if col in bdf.columns and bdf[col].notna().any():
            path_col = col
            break
    if path_col is None:
        return [], []
    
    valid = bdf[bdf[path_col].notna()]
    if len(valid) == 0:
        return [], []
    indices = np.linspace(0, len(valid) - 1, n, dtype=int)
    samples = valid.iloc[indices]
    urls, labels = [], []
    for _, row in samples.iterrows():
        url = f'{DJANGO_BASE}/{row[path_col]}'
        print(url)
        urls.append(url)
        labels.append(row['ds'].strftime('%Y-%m-%d %H:%M'))
    return urls, labels

def load_image(source):
    try:
        if isinstance(source, Path):
            return Image.open(source)
        else:
            req = Request(source, headers={'User-Agent': 'Mozilla/5.0'})
            with urlopen(req, timeout=5) as resp:
                return Image.open(BytesIO(resp.read()))
    except Exception:
        return None

def show_beach_images(beach, sources, labels, period, cls, ratio):
    n = len(sources)
    if n == 0:
        print(f'  {beach}: no images found')
        return
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, src, lbl in zip(axes, sources, labels):
        img = load_image(src)
        if img is not None:
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, 'Not found', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(lbl)
        ax.axis('off')
    ratio_str = f'{ratio:.1f}x' if not np.isnan(ratio) else 'N/A'
    fig.suptitle(f'{beach} [{period}] — {cls} | ratio={ratio_str}')
    plt.tight_layout()
    plt.show()

# Show images for all flagged beaches
if HAS_CACHE and not season_cache_df.empty:
    print('CACHE (2022-2023) — Flagged Datasets')
    print('=' * 60)
    flagged_c = season_cache_df[season_cache_df['classification'].str.contains('No signal|Insufficient')]
    for _, row in flagged_c.iterrows():
        sources, labels = get_cache_images(row['beach'], cache_all)

print()
print('=' * 60)
print('DJANGO (2025-2026) — Flagged Datasets')
print('=' * 60)
flagged_d = season_django_df[season_django_df['classification'].str.contains('No signal|Insufficient')]
for _, row in flagged_d.iterrows():
    sources, labels = get_django_images(row['beach'], django_all)
    show_beach_images(row['beach'], sources, labels, 'django', row['classification'], row['ratio'])

plt.rcParams.update(BASE_RC)


### Remove Suspect Datasets

Remove suspect **datasets** (not entire cameras) per period.
A beach can be good in 2022 but bad in 2025 — only the bad period is removed.
All subsequent analysis uses `cache_clean` and `django_clean`.

In [ ]:
FORCE_KEEP_CACHE = []

suspect_beaches_cache = set(suspect_list_c['beach']) if (HAS_CACHE and len(suspect_list_c) > 0) else set()
suspect_beaches_django = set(suspect_list_d['beach']) if len(suspect_list_d) > 0 else set()

# Cache: legacy folder exclusions (known bad sources)
EXCLUDE_FOLDERS = [
    'livecampro/001', 'livecampro/011', 'livecampro/018', 'livecampro/021',
    'livecampro/030', 'livecampro/039', 'livecampro/070', 'MultimediaTres/PortAndratx',
    'SeeTheWorld/mallorca_pancam', 'skyline/es-pujols',
]
EXCLUDE_PREFIX = ['ibred', 'ClubNauticSoller', 'Guenthoer', 'youtube']

legacy_exclude_cache = set()
if HAS_CACHE:
    legacy_mask = cache_all['beach_folder'].isin(EXCLUDE_FOLDERS)
    for p in EXCLUDE_PREFIX:
        legacy_mask |= cache_all['beach_folder'].str.startswith(p, na=False)
    legacy_exclude_cache = set(cache_all.loc[legacy_mask, 'unique_id'].unique())

# Cache: remove suspect + legacy + insufficient with no override
force_keep_c = set(FORCE_KEEP_CACHE)
remove_cache = (suspect_beaches_cache | legacy_exclude_cache) - force_keep_c

# Django: remove ONLY auto-detected suspects (no manual list)
remove_django = suspect_beaches_django

if HAS_CACHE:
    cache_clean = cache_all[~cache_all['unique_id'].isin(remove_cache)].copy()
    if suspect_beaches_cache:
        print(f'Cache:  removed {len(suspect_beaches_cache)} suspect: {sorted(suspect_beaches_cache)}')
    if legacy_exclude_cache:
        print(f'Cache:  removed {len(legacy_exclude_cache)} legacy folder excludes')
    print(f'Cache clean:  {len(cache_clean):,} records, {cache_clean["unique_id"].nunique()} cameras')
else:
    cache_clean = pd.DataFrame()

django_clean = django_all[~django_all['unique_id'].isin(remove_django)].copy()
if suspect_beaches_django:
    print(f'Django: removed {len(suspect_beaches_django)} suspect: {sorted(suspect_beaches_django)}')
print(f'Django clean: {len(django_clean):,} records, {django_clean["unique_id"].nunique()} cameras')

# Reindex
if HAS_CACHE:
    cache_clean = cache_clean.groupby('unique_id').filter(lambda x: len(x) > 0).reset_index(drop=True)
django_clean = django_clean.groupby('unique_id').filter(lambda x: len(x) > 0).reset_index(drop=True)

df_final = pd.concat([cache_clean, django_clean], ignore_index=True) if HAS_CACHE else django_clean.copy()

print(f'\nFinal: {len(df_final):,} records, {df_final["unique_id"].nunique()} cameras')
for item in sorted(df_final['unique_id'].unique()):
    src_counts = df_final[df_final['unique_id'] == item]['source'].value_counts().to_dict()
    print(f'  {item:45s} {src_counts}')

In [ ]:
# Manual cache exclusions (known bad sources we can't fix)
EXCLUDE_CACHE = [
    "Port de Soller",
]

uid_to_name = df_final.groupby('unique_id')['beach_name'].first().to_dict() if 'beach_name' in df_final.columns else {}

def resolve_patterns(patterns, df):
    matched = set()
    for p in patterns:
        for uid in df['unique_id'].unique():
            name = uid_to_name.get(uid, '')
            if re.search(p, uid, re.IGNORECASE) or re.search(p, str(name), re.IGNORECASE):
                matched.add(uid)
    return matched

exclude_c = resolve_patterns(EXCLUDE_CACHE, df_final[df_final['source'] == 'cache']) if HAS_CACHE else set()

n_before = len(df_final)
if exclude_c:
    mask = (df_final['unique_id'].isin(exclude_c)) & (df_final['source'] == 'cache')
    names = sorted(set(uid_to_name.get(u, u) for u in exclude_c))
    print(f'Cache manual exclusion: removing {mask.sum():,} records from {len(exclude_c)} cameras: {names}')
    df_final = df_final[~mask].reset_index(drop=True)
else:
    print('No manual cache exclusions applied')

print(f'Dataset: {len(df_final):,} records ({n_before - len(df_final):,} removed), {df_final["unique_id"].nunique()} cameras')

In [ ]:
if HAS_CACHE and exclude_c:
    n_cc = len(cache_clean)
    cache_clean = cache_clean[~cache_clean['unique_id'].isin(exclude_c)].reset_index(drop=True)
    print(f'cache_clean synced: {n_cc:,} → {len(cache_clean):,} ({cache_clean["unique_id"].nunique()} cameras)')

In [ ]:
plt.rcParams.update(MID_RC)
N_IMAGES = 4

for BEACH in df_final['unique_id'].unique():
    for PERIOD in ['cache', 'django']:
        bdf = df_final[(df_final['unique_id'] == BEACH) & (df_final['source'] == PERIOD)]
        if len(bdf) == 0:
            continue
        if PERIOD == 'django':
            path_col = None
            for col in ['prediction_path', 'image_path']:
                if col in bdf.columns and bdf[col].notna().any():
                    path_col = col
                    break
            if path_col is None:
                continue
            bdf = bdf[bdf[path_col].notna()]
            if len(bdf) == 0:
                continue
            samples = bdf.sample(n=min(N_IMAGES, len(bdf)), random_state=None)
            sources = [f'{DJANGO_BASE}/{row[path_col]}' for _, row in samples.iterrows()]
            labels = [f'{row["ds"].strftime("%Y-%m-%d %H:%M")}  |  y={row["y"]:.0f}' for _, row in samples.iterrows()]
        else:
            folders = raw.loc[raw['unique_id'] == BEACH, 'beach_folder'].replace('', np.nan).dropna().unique()
            if len(folders) == 0:
                continue
            folder = CACHE_ROOT / folders[0]
            imgs = sorted(folder.glob('*.jpg')) + sorted(folder.glob('*.png'))
            if len(imgs) == 0:
                continue
            selected = [imgs[i] for i in np.random.choice(len(imgs), min(N_IMAGES, len(imgs)), replace=False)]
            sources = selected
            labels = [p.stem for p in selected]

        cols = min(4, len(sources))
        rows = (len(sources) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 3.5 * rows), squeeze=False)
        axes = np.array(axes).flatten()
        for ax, src, lbl in zip(axes, sources, labels):
            img = load_image(src) if src else None
            if img is not None:
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, 'Not found', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(lbl)
            ax.axis('off')
        for j in range(len(sources), len(axes)):
            axes[j].set_visible(False)
        fig.suptitle(f'{BEACH} [{PERIOD}] — {len(sources)} random samples')
        plt.tight_layout()
        plt.show()
plt.rcParams.update(BASE_RC)


In [ ]:
plt.rcParams.update(MID_RC)
# Per-beach scatter + 30d trend — CLEAN datasets
for period_label, source_key, season_df in [
    ('Cache (2022-2023)', 'cache', season_cache_df),
    ('Django (2025-2026)', 'django', season_django_df),
]:
    subset = df_final[df_final['source'] == source_key]
    beaches = sorted(subset['unique_id'].unique())
    n_b = len(beaches)
    if n_b == 0:
        continue
    cols_p = 3
    rows_b = (n_b + cols_p - 1) // cols_p
    fig, axes = plt.subplots(rows_b, cols_p, figsize=(17, 3.5 * rows_b), sharex=False, squeeze=False)
    axes = np.array(axes).flatten()
    for i, beach in enumerate(beaches):
        bdf = subset[subset['unique_id'] == beach].sort_values('ds')
        row_info = season_df[season_df['beach'] == beach]
        cls = row_info['classification'].values[0] if len(row_info) > 0 else '?'
        ratio = row_info['ratio'].values[0] if len(row_info) > 0 else np.nan
        color = get_cls_color(cls)
        axes[i].scatter(bdf['ds'], bdf['y'], s=1, alpha=0.2, color=color, rasterized=True)
        daily = bdf.set_index('ds')['y'].resample('D').mean().dropna()
        if len(daily) > 30:
            trend = daily.rolling(30, center=True, min_periods=10).mean()
            axes[i].plot(trend.index, trend.values, color='black', lw=1.5, label='30d trend')
        if len(daily) > 10:
            x_num = (daily.index - daily.index[0]).days.values.astype(float)
            slope, intercept = np.polyfit(x_num, daily.values, 1)
            axes[i].plot(daily.index, intercept + slope * x_num,
                         color='red', lw=1.2, ls='--', label=f'slope={slope:.2f}/day')
        s_mean = bdf[bdf['month'].isin(summer_months)]['y'].mean()
        w_mean = bdf[bdf['month'].isin(winter_months)]['y'].mean()
        if not np.isnan(s_mean):
            axes[i].axhline(s_mean, color='#e74c3c', ls=':', lw=1, alpha=0.5, label=f'summer={s_mean:.0f}')
        if not np.isnan(w_mean):
            axes[i].axhline(w_mean, color='#2980b9', ls=':', lw=1, alpha=0.5, label=f'winter={w_mean:.0f}')
        ratio_str = f'{ratio:.1f}x' if not np.isnan(ratio) else 'N/A'
        axes[i].set_title(f"{beach}\n{cls} | ratio={ratio_str}", pad=3)
        axes[i].set_ylabel('y')
        axes[i].tick_params(axis='x', rotation=30, labelsize=6)
        axes[i].tick_params(axis='y', labelsize=7)
        axes[i].legend(fontsize=5, loc='upper right', ncol=2)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    fig.suptitle(f'{period_label} — Clean Dataset Scatter + Trend', y=1.02)
    plt.tight_layout()
    plt.show()
plt.rcParams.update(BASE_RC)


## 1.4 Temporal Patterns

In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    
    hourly_c = cache_clean.groupby('hour')['y'].agg(['mean', 'median', 'std'])
    axes[0, 0].fill_between(hourly_c.index, hourly_c['mean'] - hourly_c['std'],
                             hourly_c['mean'] + hourly_c['std'], alpha=0.15)
    axes[0, 0].plot(hourly_c.index, hourly_c['mean'], 'o-', ms=5, label='mean')
    axes[0, 0].plot(hourly_c.index, hourly_c['median'], 's--', ms=4, alpha=0.7, label='median')
    axes[0, 0].set_title('Cache — Occupancy by Hour')
    axes[0, 0].set_xlabel('Hour')
    axes[0, 0].set_ylabel('Crowd count')
    axes[0, 0].legend()
    
    dow_c = cache_clean.groupby('day_of_week')['y'].mean()
    dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    colors_dow = ['#e74c3c' if d >= 5 else '#3498db' for d in dow_c.index]
    axes[0, 1].bar(dow_c.index, dow_c.values, color=colors_dow, alpha=0.8)
    axes[0, 1].set_xticks(range(7))
    axes[0, 1].set_xticklabels(dow_labels)
    axes[0, 1].set_title('Cache — Day of Week')
    axes[0, 1].set_ylabel('Mean y')
    
    monthly_c = cache_clean.groupby('month')['y'].agg(['mean', 'std'])
    axes[1, 0].bar(monthly_c.index, monthly_c['mean'], color='#f39c12', alpha=0.8, edgecolor='white')
    axes[1, 0].errorbar(monthly_c.index, monthly_c['mean'], yerr=monthly_c['std'],
                         fmt='none', color='black', capsize=3, alpha=0.5)
    axes[1, 0].set_title('Cache — Monthly')
    axes[1, 0].set_xlabel('Month')
    axes[1, 0].set_ylabel('Mean y')
    axes[1, 0].set_xticks(range(1, 13))
    
    pivot_c = cache_clean.pivot_table(values='y', index='unique_id', columns='hour', aggfunc='mean')
    if not pivot_c.empty and pivot_c.notna().any().any():
        sns.heatmap(pivot_c, cmap='YlOrRd', ax=axes[1, 1], linewidths=0.3,
                    cbar_kws={'label': 'Mean count'})
    else:
        axes[1, 1].text(0.5, 0.5, 'No data', ha='center', va='center', transform=axes[1, 1].transAxes)
    axes[1, 1].set_title('Cache — Beach × Hour')
    
    plt.tight_layout()
    plt.show()
plt.rcParams.update(BASE_RC)


In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE:
    # Weekend vs weekday
    fig, ax = plt.subplots(figsize=(10, 4))
    for val, label in [(0, 'Weekday'), (1, 'Weekend')]:
        grp = cache_clean[cache_clean['is_weekend'] == val].groupby('hour')['y'].mean()
        ax.plot(grp.index, grp.values, 'o-', ms=5, label=label)
    ax.set_xlabel('Hour')
    ax.set_ylabel('Mean y')
    ax.set_title('Cache — Weekday vs Weekend')
    ax.legend()
    plt.tight_layout()
    plt.show()
plt.rcParams.update(BASE_RC)


## 1.6 Weather & Correlations

In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE:
    avail_c = {k: v for k, v in key_w.items() if k in cache_clean.columns and cache_clean[k].notna().sum() > 0}
    if avail_c:
        fig, axes = plt.subplots(2, 3, figsize=(15, 7), squeeze=False)
        axes = axes.flatten()
        for i, (col, label) in enumerate(avail_c.items()):
            cache_clean[col].dropna().hist(bins=40, ax=axes[i], alpha=0.8, edgecolor='white')
            axes[i].set_title(label)
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)
        plt.suptitle('Cache — Weather Distributions')
        plt.tight_layout()
        plt.show()
plt.rcParams.update(BASE_RC)


In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE:
    # Weather vs y scatter
    scatter_cols = [c for c in ['om_temperature_2m', 'om_cloud_cover',
                                 'om_wind_speed_10m', 'om_shortwave_radiation']
                    if c in cache_clean.columns and cache_clean[c].notna().sum() > 100]
    
    if scatter_cols:
        fig, axes = plt.subplots(1, len(scatter_cols), figsize=(5 * len(scatter_cols), 4))
        if len(scatter_cols) == 1:
            axes = [axes]
        for i, col in enumerate(scatter_cols):
            sub = cache_clean[[col, 'y']].dropna()
            axes[i].scatter(sub[col], sub['y'], alpha=0.03, s=2, rasterized=True)
            bins = pd.cut(sub[col], bins=20)
            trend = sub.groupby(bins, observed=True)['y'].mean()
            x_mid = [(b.left + b.right) / 2 for b in trend.index]
            axes[i].plot(x_mid, trend.values, 'r-o', ms=4, lw=2)
            axes[i].set_xlabel(col.replace('om_', '').replace('_', ' '))
            axes[i].set_ylabel('y')
        plt.suptitle('Cache — Weather → Occupancy')
        plt.tight_layout()
        plt.show()
plt.rcParams.update(BASE_RC)


In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE:
    # Correlation with y
    feats_c = ['hour', 'day_of_week', 'month', 'is_weekend'] + [c for c in weather_cols if c in cache_clean.columns]
    feats_c = [c for c in feats_c if cache_clean[c].notna().sum() > 100]
    corr_c = cache_clean[feats_c + ['y']].corr()['y'].drop('y').sort_values()
    
    fig, ax = plt.subplots(figsize=(8, max(4, len(corr_c) * 0.22)))
    colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr_c]
    if not corr_c.empty:
        corr_c.plot.barh(ax=ax, color=colors, alpha=0.8)
    ax.set_title('Cache — Correlation with y')
    ax.axvline(0, color='black', lw=0.5)
    ax.set_xlabel('Pearson r')
    plt.tight_layout()
    plt.show()
plt.rcParams.update(BASE_RC)


## 1.7 Data Gaps

In [ ]:
if HAS_CACHE:
    gap_c = []
    c_clean_beaches = sorted(cache_clean['unique_id'].unique())
    for beach in c_clean_beaches:
        bdf = cache_clean[cache_clean['unique_id'] == beach].sort_values('ds')
        diffs_h = bdf['ds'].diff().dt.total_seconds().dropna() / 3600
        gap_c.append({
            'beach': beach, 'records': len(bdf),
            'range': f"{bdf['ds'].min().date()} → {bdf['ds'].max().date()}",
            'median_step_h': round(diffs_h.median(), 1) if len(diffs_h) > 0 else None,
            'gaps_>24h': int((diffs_h > 24).sum()),
            'max_gap_d': round(diffs_h.max() / 24, 1) if len(diffs_h) > 0 else 0,
        })
    
    pd.DataFrame(gap_c).sort_values('records', ascending=False)

---
# Part 2 — Django Period (2025-2026)

Production data from the Django web application.

## 2.1 Overview

In [ ]:
print(f"Shape:      {django_clean.shape}")
print(f"Date range: {django_clean['ds'].min()} → {django_clean['ds'].max()}")
print(f"Beaches:    {django_clean['unique_id'].nunique()}")
print(f"Open-Meteo: {django_clean[om_cols].notna().all(axis=1).mean()*100:.1f}% complete")

django_clean_summary = django_clean.groupby('unique_id').agg(
    n=('y', 'size'), mean=('y', 'mean'), std=('y', 'std'),
    min=('y', 'min'), max=('y', 'max'),
    first=('ds', 'min'), last=('ds', 'max'),
).round(1)
django_clean_summary['cv'] = (django_clean_summary['std'] / django_clean_summary['mean']).round(3)
django_clean_summary.sort_values('n', ascending=False)

## 2.2 Crowd Count Distribution

In [ ]:
plt.rcParams.update(MID_RC)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

django_clean['y'].hist(bins=60, ax=axes[0], color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(django_clean['y'].median(), color='red', ls='--',
                label=f"median={django_clean['y'].median():.0f}")
axes[0].set_title('Django — Distribution of y')
axes[0].set_xlabel('Crowd count')
axes[0].legend()

np.log1p(django_clean['y']).hist(bins=60, ax=axes[1], color='#9b59b6', edgecolor='white', alpha=0.8)
axes[1].set_title('Django — log(1+y)')

order_d = django_clean.groupby('unique_id')['y'].median().sort_values().index
sns.boxplot(data=django_clean, y='unique_id', x='y', order=order_d, ax=axes[2],
            fliersize=1, linewidth=0.5)
axes[2].set_title('Django — y by Beach')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

print(django_clean['y'].describe().round(2))
print(f"Skewness: {django_clean['y'].skew():.2f}")
plt.rcParams.update(BASE_RC)


## 2.4 Temporal Patterns

In [ ]:
plt.rcParams.update(MID_RC)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

hourly_d = django_clean.groupby('hour')['y'].agg(['mean', 'median', 'std'])
axes[0, 0].fill_between(hourly_d.index, hourly_d['mean'] - hourly_d['std'],
                         hourly_d['mean'] + hourly_d['std'], alpha=0.15, color='C1')
axes[0, 0].plot(hourly_d.index, hourly_d['mean'], 'o-', ms=5, color='C1', label='mean')
axes[0, 0].plot(hourly_d.index, hourly_d['median'], 's--', ms=4, alpha=0.7, color='C1', label='median')
axes[0, 0].set_title('Django — Occupancy by Hour')
axes[0, 0].set_xlabel('Hour')
axes[0, 0].set_ylabel('Crowd count')
axes[0, 0].legend()

dow_d = django_clean.groupby('day_of_week')['y'].mean()
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
colors_dow = ['#e74c3c' if d >= 5 else '#3498db' for d in dow_d.index]
axes[0, 1].bar(dow_d.index, dow_d.values, color=colors_dow, alpha=0.8)
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(dow_labels)
axes[0, 1].set_title('Django — Day of Week')
axes[0, 1].set_ylabel('Mean y')

monthly_d = django_clean.groupby('month')['y'].agg(['mean', 'std'])
axes[1, 0].bar(monthly_d.index, monthly_d['mean'], color='#e67e22', alpha=0.8, edgecolor='white')
axes[1, 0].errorbar(monthly_d.index, monthly_d['mean'], yerr=monthly_d['std'],
                     fmt='none', color='black', capsize=3, alpha=0.5)
axes[1, 0].set_title('Django — Monthly')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Mean y')
axes[1, 0].set_xticks(range(1, 13))

pivot_d = django_clean.pivot_table(values='y', index='unique_id', columns='hour', aggfunc='mean')
if not pivot_d.empty and pivot_d.notna().any().any():
    sns.heatmap(pivot_d, cmap='YlOrRd', ax=axes[1, 1], linewidths=0.3,
                cbar_kws={'label': 'Mean count'})
else:
    axes[1, 1].text(0.5, 0.5, 'No data', ha='center', va='center', transform=axes[1, 1].transAxes)
axes[1, 1].set_title('Django — Beach × Hour')

plt.tight_layout()
plt.show()
plt.rcParams.update(BASE_RC)


In [ ]:
plt.rcParams.update(MID_RC)
# Weekend vs weekday
fig, ax = plt.subplots(figsize=(10, 4))
for val, label in [(0, 'Weekday'), (1, 'Weekend')]:
    grp = django_clean[django_clean['is_weekend'] == val].groupby('hour')['y'].mean()
    ax.plot(grp.index, grp.values, 'o-', ms=5, label=label)
ax.set_xlabel('Hour')
ax.set_ylabel('Mean y')
ax.set_title('Django — Weekday vs Weekend')
ax.legend()
plt.tight_layout()
plt.show()
plt.rcParams.update(BASE_RC)


## 2.6 Weather & Correlations

In [ ]:
plt.rcParams.update(MID_RC)
avail_d = {k: v for k, v in key_w.items() if k in django_clean.columns and django_clean[k].notna().sum() > 0}
if avail_d:
    fig, axes = plt.subplots(2, 3, figsize=(15, 7), squeeze=False)
    axes = axes.flatten()
    for i, (col, label) in enumerate(avail_d.items()):
        django_clean[col].dropna().hist(bins=40, ax=axes[i], alpha=0.8, edgecolor='white', color='C1')
        axes[i].set_title(label)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Django — Weather Distributions')
    plt.tight_layout()
    plt.show()
plt.rcParams.update(BASE_RC)


In [ ]:
plt.rcParams.update(MID_RC)
feats_d = ['hour', 'day_of_week', 'month', 'is_weekend'] + [c for c in weather_cols if c in django_clean.columns]
feats_d = [c for c in feats_d if django_clean[c].notna().sum() > 100]
corr_d = django_clean[feats_d + ['y']].corr()['y'].drop('y').sort_values()

fig, ax = plt.subplots(figsize=(8, max(4, len(corr_d) * 0.22)))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr_d]
corr_d.plot.barh(ax=ax, color=colors, alpha=0.8)
ax.set_title('Django — Correlation with y')
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()
plt.rcParams.update(BASE_RC)


## 2.7 Data Gaps

In [ ]:
gap_d = []
d_clean_beaches = sorted(django_clean['unique_id'].unique())
for beach in d_clean_beaches:
    bdf = django_clean[django_clean['unique_id'] == beach].sort_values('ds')
    diffs_h = bdf['ds'].diff().dt.total_seconds().dropna() / 3600
    gap_d.append({
        'beach': beach, 'records': len(bdf),
        'range': f"{bdf['ds'].min().date()} → {bdf['ds'].max().date()}",
        'median_step_h': round(diffs_h.median(), 1) if len(diffs_h) > 0 else None,
        'gaps_>24h': int((diffs_h > 24).sum()),
        'max_gap_d': round(diffs_h.max() / 24, 1) if len(diffs_h) > 0 else 0,
    })

pd.DataFrame(gap_d).sort_values('records', ascending=False)

---
# Part 3 — Cross-Period Comparison

Comparing cache (2022-2023) vs Django (2025-2026) to detect magnitude shifts,
camera changes, and assess whether the two periods can be merged for TFT training.

## 3.1 Beach Overlap

In [ ]:
if HAS_CACHE:
    cache_set = set(cache_clean['unique_id'].unique())
    django_set = set(django_clean['unique_id'].unique())
    both_set = cache_set & django_set
    cache_only = cache_set - django_set
    django_only = django_set - cache_set
    
    print(f'Beaches in both periods:     {len(both_set)}')
    print(f'Cache only (2022):           {len(cache_only)}')
    print(f'Django only (2025):          {len(django_only)}')
    
    if cache_only:
        print(f'\nCache-only beaches: {sorted(cache_only)}')
    if django_only:
        print(f'\nDjango-only beaches: {sorted(django_only)}')
    

## 3.2 Magnitude Shift Detection

In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE:
    shift_rows = []
    for beach in sorted(both_set):
        c_y = cache_clean[cache_clean['unique_id'] == beach]['y']
        d_y = django_clean[django_clean['unique_id'] == beach]['y']
        if len(c_y) < 10 or len(d_y) < 10:
            continue
        ratio = c_y.mean() / d_y.mean() if d_y.mean() > 0 else np.nan
        shift_rows.append({
            'beach': beach,
            'cache_mean': round(c_y.mean(), 1), 'cache_n': len(c_y),
            'django_mean': round(d_y.mean(), 1), 'django_n': len(d_y),
            'ratio': round(ratio, 2),
        })

    shift_df = pd.DataFrame(shift_rows)
    if shift_df.empty:
        print('No shared beaches with enough data for magnitude comparison')
    else:
        shift_df = shift_df.sort_values('ratio', ascending=False)

        fig, ax = plt.subplots(figsize=(10, max(4, len(shift_df) * 0.35)))
        colors = ['#e74c3c' if abs(r - 1) > 2 else '#f39c12' if abs(r - 1) > 0.5 else '#2ecc71'
                  for r in shift_df['ratio'].fillna(1)]
        if not shift_df.empty:
            ax.barh(shift_df['beach'], shift_df['ratio'], color=colors, alpha=0.8)
        ax.axvline(1.0, color='black', ls='-', lw=1)
        ax.axvline(3.0, color='red', ls='--', lw=1, label='3× shift threshold')
        ax.axvline(1/3, color='red', ls='--', lw=1)
        ax.set_title('Mean y Ratio: Cache / Django (1.0 = same magnitude)')
        ax.set_xlabel('cache_mean / django_mean')
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

        shifted = shift_df[(shift_df['ratio'] > 3) | (shift_df['ratio'] < 1/3)]
        if len(shifted) > 0:
            print(f"⚠️  {len(shifted)} beaches with >3× magnitude shift:")
            print(shifted.to_string(index=False))
        else:
            print('✓ No major magnitude shifts detected.')
plt.rcParams.update(BASE_RC)


In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE and not shift_df.empty:
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(shift_df['cache_mean'], shift_df['django_mean'], s=50, alpha=0.7, edgecolors='black', lw=0.5)
    for _, row in shift_df.iterrows():
        ax.annotate(row['beach'], (row['cache_mean'], row['django_mean']),
                    fontsize=6, xytext=(3, 3), textcoords='offset points')

    lims = [0, max(shift_df[['cache_mean', 'django_mean']].max().max() * 1.1, 1)]
    ax.plot(lims, lims, 'r--', alpha=0.5, label='1:1 (identical)')
    ax.set_xlabel('Cache mean y (2022-2023)')
    ax.set_ylabel('Django mean y (2025-2026)')
    ax.set_title('Per-Beach Mean Occupancy: Cache vs Django')
    ax.legend()
    plt.tight_layout()
    plt.show()
plt.rcParams.update(BASE_RC)


## 3.3 Hourly Profile Comparison

In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE:
    # Global hourly curves: cache vs django
    fig, axes = plt.subplots(figsize=(14, 4))
    
    # Absolute
    for src, label, color in [('cache', 'Cache (2022-23)', 'C0'), ('django', 'Django (2025-26)', 'C1')]:
        grp = df_final[df_final['source'] == src].groupby('hour')['y'].mean()
        axes.plot(grp.index, grp.values, 'o-', ms=5, label=label, color=color)
    axes.set_title('Mean y by Hour — Absolute')
    axes.set_xlabel('Hour')
    axes.set_ylabel('Mean y')
    axes.legend()
    
    
    plt.tight_layout()
    plt.show()
plt.rcParams.update(BASE_RC)


In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE:
    shared = sorted(both_set)
    if not shared:
        print('No shared beaches between cache and django')
    else:
        n_s = min(len(shared), 12)
        cols_s = 3
        rows_s = (n_s + cols_s - 1) // cols_s

        fig, axes = plt.subplots(rows_s, cols_s, figsize=(16, 3.5 * rows_s), sharex=True, squeeze=False)
        axes = np.array(axes).flatten()

        for i, beach in enumerate(shared[:n_s]):
            for src, color, label in [('cache', 'C0', 'Cache'), ('django', 'C1', 'Django')]:
                bdf = df_final[(df_final['unique_id'] == beach) & (df_final['source'] == src)]
                if len(bdf) < 10:
                    continue
                hourly = bdf.groupby('hour')['y'].mean()
                axes[i].plot(hourly.index, hourly.values, 'o-', ms=3, color=color, label=label)
            axes[i].set_title(beach)
            axes[i].set_ylabel('Mean y')
            if i == 0:
                axes[i].legend(fontsize=7)

        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)

        fig.suptitle('Hourly Profile by Beach: Cache vs Django', y=1.01)
        plt.tight_layout()
        plt.show()
plt.rcParams.update(BASE_RC)


## 3.5 Weather Comparison

In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE:
    compare_w = ['om_temperature_2m', 'om_cloud_cover', 'om_wind_speed_10m', 'om_shortwave_radiation']
    compare_w = [c for c in compare_w if c in df_final.columns]
    
    if compare_w:
        fig, axes = plt.subplots(1, len(compare_w), figsize=(5 * len(compare_w), 4))
        if len(compare_w) == 1:
            axes = [axes]
        for i, col in enumerate(compare_w):
            for src, color, label in [('cache', 'C0', 'Cache'), ('django', 'C1', 'Django')]:
                vals = df_final.loc[df_final['source'] == src, col].dropna()
                axes[i].hist(vals, bins=40, alpha=0.5, color=color, label=label, density=True)
            axes[i].set_title(col.replace('om_', '').replace('_', ' '))
            axes[i].legend(fontsize=8)
        plt.suptitle('Weather Distribution: Cache vs Django')
        plt.tight_layout()
        plt.show()
plt.rcParams.update(BASE_RC)


## 3.6 Correlation Comparison

In [ ]:
plt.rcParams.update(MID_RC)
if HAS_CACHE:
    # Side-by-side correlation with y: cache vs django
    shared_feats = [c for c in feats_c if c in feats_d and c in df.columns]
    
    corr_compare = pd.DataFrame({
        'cache': cache_clean[shared_feats + ['y']].corr()['y'].drop('y'),
        'django': django_clean[shared_feats + ['y']].corr()['y'].drop('y'),
    }).sort_values('cache')
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(corr_compare) * 0.28)))
    y_pos = np.arange(len(corr_compare))
    ax.barh(y_pos - 0.15, corr_compare['cache'], height=0.3, color='C0', alpha=0.8, label='Cache')
    ax.barh(y_pos + 0.15, corr_compare['django'], height=0.3, color='C1', alpha=0.8, label='Django')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(corr_compare.index, fontsize=8)
    ax.axvline(0, color='black', lw=0.5)
    ax.set_title('Correlation with y: Cache vs Django')
    ax.set_xlabel('Pearson r')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    # Biggest differences
    corr_compare['diff'] = (corr_compare['cache'] - corr_compare['django']).abs()
    print('Features with biggest correlation difference between periods:')
    print(corr_compare.nlargest(5, 'diff').round(3).to_string())
plt.rcParams.update(BASE_RC)


## 3.7 Coverage Timeline

In [ ]:
plt.rcParams.update(MID_RC)
all_beaches = sorted(df_final['unique_id'].unique())

fig, ax = plt.subplots(figsize=(16, max(4, len(all_beaches) * 0.35)))

for i, beach in enumerate(all_beaches):
    for src, color in [('cache', 'C0'), ('django', 'C1')]:
        bdf = df[(df_final['unique_id'] == beach) & (df['source'] == src)]
        if len(bdf) == 0:
            continue
        dates = bdf['ds'].dt.date.unique()
        ax.scatter([pd.Timestamp(d) for d in dates], [i] * len(dates),
                   s=1.5, alpha=0.4, color=color, rasterized=True)

ax.set_yticks(range(len(all_beaches)))
ax.set_yticklabels(all_beaches, fontsize=6)
ax.set_title('Full Coverage Timeline (blue=cache, orange=django)')
ax.set_xlabel('Date')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
plt.rcParams.update(BASE_RC)


---
## 3.8 Holiday Effect Analysis

Compare beach occupancy on holidays vs normal days to quantify the holiday effect.
Uses Spanish national + Balearic Islands regional holidays + bridge days (puentes).


In [ ]:
import holidays as _holidays
from datetime import timedelta as _td

if 'is_holiday' not in df_final.columns:
    years = sorted(df_final['ds'].dt.year.unique())
    es_h = _holidays.Spain(prov='IB', years=years)
    bridges = {}
    for d, name in es_h.items():
        dow = d.weekday()
        if dow == 3:
            bridges[d + _td(days=1)] = f'Puente: {name}'
        elif dow == 1:
            bridges[d - _td(days=1)] = f'Puente: {name}'
    all_special = {pd.Timestamp(k): v for k, v in es_h.items()}
    all_special.update({pd.Timestamp(k): v for k, v in bridges.items()})
    dates = df_final['ds'].dt.normalize()
    df_final['holiday_name'] = dates.map(all_special).fillna('')
    df_final['is_holiday'] = (df_final['holiday_name'] != '').astype(int)
    print('Created is_holiday / holiday_name columns')

hol = df_final[df_final['is_holiday'] == 1]
print(f'Holiday records: {len(hol):,} / {len(df_final):,} ({100*len(hol)/len(df_final):.1f}%)')
print(f'Unique holidays: {hol["holiday_name"].nunique()}')
print()
print(hol.groupby('holiday_name').agg(
    records=('y', 'size'), mean_y=('y', 'mean'), median_y=('y', 'median')
).sort_values('records', ascending=False).round(1).to_string())


In [ ]:
plt.rcParams.update(MID_RC)
# Compare distributions: Holiday vs Weekday vs Weekend
df_final['day_type'] = 'Weekday'
df_final.loc[df_final['is_weekend'] == 1, 'day_type'] = 'Weekend'
df_final.loc[df_final['is_holiday'] == 1, 'day_type'] = 'Holiday'

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Box plot
order = ['Weekday', 'Weekend', 'Holiday']
colors = {'Weekday': '#3498db', 'Weekend': '#2ecc71', 'Holiday': '#e74c3c'}
bp = df_final.boxplot(column='y', by='day_type', ax=axes[0], positions=[0,1,2],
                       widths=0.6, patch_artist=True, return_type='dict')
for patch, label in zip(bp['y']['boxes'], order):
    patch.set_facecolor(colors[label])
    patch.set_alpha(0.7)
axes[0].set_title('Occupancy Distribution')
axes[0].set_xlabel('')
axes[0].set_ylabel('Crowd Count')
axes[0].set_xticklabels(order)
fig.suptitle('')

# Mean by day type
means = df_final.groupby('day_type')['y'].mean().reindex(order)
bars = axes[1].bar(order, means, color=[colors[o] for o in order], alpha=0.8, edgecolor='white')
for bar, val in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}',
                 ha='center', va='bottom', fontsize=10)
axes[1].set_title('Mean Occupancy')
axes[1].set_ylabel('Mean Crowd Count')

# Overlapping histograms
for dt in order:
    sub = df_final[df_final['day_type'] == dt]['y']
    axes[2].hist(sub, bins=50, alpha=0.5, label=f'{dt} (n={len(sub):,})',
                 color=colors[dt], density=True)
axes[2].set_title('Occupancy Density')
axes[2].set_xlabel('Crowd Count')
axes[2].legend()

plt.tight_layout()
plt.show()

# Percentage uplift
wkday_mean = df_final[df_final['day_type'] == 'Weekday']['y'].mean()
wkend_mean = df_final[df_final['day_type'] == 'Weekend']['y'].mean()
hol_mean = df_final[df_final['day_type'] == 'Holiday']['y'].mean()
print(f'Weekend uplift vs weekday: {100*(wkend_mean/wkday_mean - 1):+.1f}%')
print(f'Holiday uplift vs weekday: {100*(hol_mean/wkday_mean - 1):+.1f}%')
print(f'Holiday uplift vs weekend: {100*(hol_mean/wkend_mean - 1):+.1f}%')

plt.rcParams.update(BASE_RC)


In [ ]:
# Thesis figure: monthly occupancy by day type + holiday days per month. IEEE \textwidth (serif 8pt).
import matplotlib.pyplot as plt, pandas as pd
from pathlib import Path
_RC = {'font.family':'serif','font.serif':['Times New Roman','DejaVu Serif'],'mathtext.fontset':'stix',
       'font.size':8,'axes.titlesize':8,'axes.labelsize':8,'xtick.labelsize':7,'ytick.labelsize':7,
       'legend.fontsize':7,'axes.linewidth':0.6,'lines.linewidth':1.2}
_FIG = Path.home()/'Desktop/master_activities/TFM/beach_counting/TFM_organized/MUSI___TFM_Template/figures'
_d = df_final.copy()
_d['day_type']='Weekday'; _d.loc[_d['is_weekend']==1,'day_type']='Weekend'; _d.loc[_d['is_holiday']==1,'day_type']='Holiday'
_colors={'Weekday':'#3498db','Weekend':'#2ecc71','Holiday':'#e74c3c'}
_hol=_d[_d['is_holiday']==1]
_m=_d.groupby(['month','day_type'])['y'].mean().unstack().reindex(range(1,13))
with plt.rc_context(_RC):
    fig, axes = plt.subplots(1, 2, figsize=(7.139, 2.5))
    for dt,c in _colors.items():
        if dt in _m.columns: axes[0].plot(_m.index,_m[dt],'o-',color=c,label=dt,lw=1.4,ms=4)
    axes[0].set_xlabel('Month'); axes[0].set_ylabel('Mean crowd count'); axes[0].set_title('Monthly occupancy by day type'); axes[0].legend(); axes[0].set_xticks(range(1,13))
    _hpm=_hol.groupby('month').agg(ud=('ds',lambda x: pd.to_datetime(x).dt.date.nunique()))
    axes[1].bar(_hpm.index,_hpm['ud'],color='#e74c3c',alpha=0.7)
    for _mo,_r in _hpm.iterrows(): axes[1].text(_mo,_r['ud']+0.1,str(int(_r['ud'])),ha='center',fontsize=6)
    axes[1].set_xlabel('Month'); axes[1].set_ylabel('Holiday days'); axes[1].set_title('Holiday days per month'); axes[1].set_xticks(range(1,13))
    fig.tight_layout(); fig.savefig(_FIG/'fig_monthly_by_daytype.png', dpi=300, bbox_inches='tight'); plt.show()


In [ ]:
plt.rcParams.update({'font.size': 16, 'axes.titlesize': 18, 'axes.labelsize': 16, 'ytick.labelsize': 12, 'xtick.labelsize': 14, 'legend.fontsize': 13})
# Holiday effect — SUMMER ONLY (Jun-Sep)
# Off-season holidays (Christmas, Easter) skew the distribution because
SEASON_MONTHS = [5, 6, 7, 8, 9]
season = df_final[df_final['month'].isin(SEASON_MONTHS)].copy()
print(f'Season filter (months {SEASON_MONTHS}): {len(season):,} / {len(df_final):,} records')
print(f'  Holidays in season: {season["is_holiday"].sum():,}')
print(f'  Unique: {season[season["is_holiday"]==1]["holiday_name"].unique().tolist()}')

season['day_type'] = 'Weekday'
season.loc[season['is_weekend'] == 1, 'day_type'] = 'Weekend'
season.loc[season['is_holiday'] == 1, 'day_type'] = 'Holiday'

order = ['Weekday', 'Weekend', 'Holiday']
colors_dt = {'Weekday': '#3498db', 'Weekend': '#2ecc71', 'Holiday': '#e74c3c'}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Holiday Effect — Season Only (May–Sep)')

# Box plot
bp = season.boxplot(column='y', by='day_type', ax=axes[0,0], positions=[0,1,2],
                    widths=0.6, patch_artist=True, return_type='dict')
for patch, label in zip(bp['y']['boxes'], order):
    patch.set_facecolor(colors_dt[label])
    patch.set_alpha(0.7)
axes[0,0].set_title('Distribution')
axes[0,0].set_xlabel('')
axes[0,0].set_ylabel('Crowd Count')
axes[0,0].set_xticklabels(order)
fig.texts = [t for t in fig.texts if 'Boxplot' not in t.get_text()]

# Mean bar
means = season.groupby('day_type')['y'].mean().reindex(order)
bars = axes[0,1].bar(order, means, color=[colors_dt[o] for o in order], alpha=0.8, edgecolor='white')
for bar, val in zip(bars, means):
    axes[0,1].text(bar.get_x() + bar.get_width()/2, val + 0.5, f'{val:.1f}',
                   ha='center', va='bottom', fontsize=10)
axes[0,1].set_title('Mean Occupancy')
axes[0,1].set_ylabel('Crowd Count')

# Hourly profile
hourly_s = season.groupby(['hour', 'day_type'])['y'].mean().unstack()
for dt, color in colors_dt.items():
    if dt in hourly_s.columns:
        axes[1,0].plot(hourly_s.index, hourly_s[dt], 'o-', color=color, label=dt, lw=2)
axes[1,0].set_xlabel('Hour')
axes[1,0].set_ylabel('Mean Crowd Count')
axes[1,0].set_title('Hourly Profile')
axes[1,0].legend()
axes[1,0].set_xticks(range(8, 21))

# Per-beach uplift (season only)
bstats_s = []
for beach in sorted(season['unique_id'].unique()):
    bdf = season[season['unique_id'] == beach]
    wk = bdf[bdf['day_type'] == 'Weekday']['y']
    we = bdf[bdf['day_type'] == 'Weekend']['y']
    ho = bdf[bdf['day_type'] == 'Holiday']['y']
    if len(wk) == 0 or wk.mean() == 0:
        continue
    bstats_s.append({
        'beach': beach,
        'wk_mean': wk.mean(),
        'we_mean': we.mean() if len(we) > 0 else np.nan,
        'ho_mean': ho.mean() if len(ho) > 0 else np.nan,
        'n_hol': len(ho),
        'we_uplift': 100 * (we.mean() / wk.mean() - 1) if len(we) > 0 else np.nan,
        'ho_uplift': 100 * (ho.mean() / wk.mean() - 1) if len(ho) > 0 else np.nan,
    })
bstats_s = pd.DataFrame(bstats_s).sort_values('ho_uplift', ascending=False)
y_pos = range(len(bstats_s))
axes[1,1].barh(y_pos, bstats_s['we_uplift'], height=0.4, color='#2ecc71', alpha=0.7, label='Weekend')
axes[1,1].barh([y + 0.4 for y in y_pos], bstats_s['ho_uplift'], height=0.4, color='#e74c3c', alpha=0.7, label='Holiday')
axes[1,1].set_yticks([y + 0.2 for y in y_pos])
axes[1,1].set_yticklabels(bstats_s['beach'], fontsize=7)
axes[1,1].axvline(0, color='k', ls='--', lw=0.5)
axes[1,1].set_xlabel('% Uplift vs Weekday')
axes[1,1].set_title('Per-Beach Uplift (season)')
axes[1,1].legend(fontsize=8)
axes[1,1].invert_yaxis()

plt.tight_layout()
plt.show()

# Summary
wk_m = season[season['day_type'] == 'Weekday']['y'].mean()
we_m = season[season['day_type'] == 'Weekend']['y'].mean()
ho_m = season[season['day_type'] == 'Holiday']['y'].mean()
print(f'Season means — Weekday: {wk_m:.1f}, Weekend: {we_m:.1f}, Holiday: {ho_m:.1f}')
print(f'Weekend uplift vs weekday: {100*(we_m/wk_m - 1):+.1f}%')
print(f'Holiday uplift vs weekday: {100*(ho_m/wk_m - 1):+.1f}%')
print(f'Holiday uplift vs weekend: {100*(ho_m/we_m - 1):+.1f}%')

plt.rcParams.update(BASE_RC)


---
## 3.9 Beach Metadata (Static Features)

Beach attributes from platgesdebalears.com encoded as TFT `stat_exog` features.
Only features with predictive value for occupancy are kept:
occupancy grade, urban proximity, composition, bathing conditions, promenade, user type.

In [ ]:
STAT_FEATURES = [
    'stat_grado_de_ocupacion', 'stat_proximidad_al_nucleo_urbano',
    'stat_composicion_de_la_playa', 'stat_condiciones_de_bano',
    'stat_paseo_maritimo', 'stat_tipo_de_usuario_local', 'stat_tipo_de_usuario_turista'
]
stat_cols = [c for c in STAT_FEATURES if c in df_final.columns]

if stat_cols:
    meta_df = df_final.groupby('unique_id')[stat_cols + ['lat', 'lon']].first().reset_index()
    has_meta = meta_df[stat_cols].notna().any(axis=1)
    print(f'{len(stat_cols)} static features, {has_meta.sum()}/{len(meta_df)} beaches with metadata\n')

    missing = meta_df[~has_meta][['unique_id', 'lat', 'lon']]
    if len(missing):
        print(f'Beaches WITHOUT metadata ({len(missing)}):')
        print('  (no profile matched within --threshold-km in build_dataset_v3.py)')
        for _, r in missing.iterrows():
            print(f'  {r["unique_id"]:45s} lat={r["lat"]:.4f}  lon={r["lon"]:.4f}')
        print()

    display(meta_df.style.map(
        lambda v: 'background-color: #ffdddd' if pd.isna(v) else '',
        subset=stat_cols
    ))
else:
    print('No stat_* columns in dataset.')
    print(f'\nAvailable columns: {[c for c in df_final.columns if "stat" in c.lower()]}')

In [ ]:
plt.rcParams.update(MID_RC)
# Coverage heatmap
if stat_cols:
    coverage = meta_df.set_index('unique_id')[stat_cols].notna().astype(int)
else:
    coverage = pd.DataFrame()

if not coverage.empty:
    fig, ax = plt.subplots(figsize=(10, max(4, len(coverage) * 0.4)))
    if coverage.notna().any().any():
        sns.heatmap(coverage, cmap='RdYlGn', cbar_kws={'label': 'Has data'},
                linewidths=0.5, ax=ax, vmin=0, vmax=1)
    ax.set_title('Beach Metadata Coverage')
    ax.set_ylabel('Beach')
    plt.tight_layout()
    plt.show()
    pct = coverage.mean() * 100
    print('Coverage per field:')
    for col in pct.sort_values(ascending=False).index:
        print(f'  {col:45s} {pct[col]:.0f}%')
else:
    print('No metadata to visualize')

plt.rcParams.update(BASE_RC)


In [ ]:
plt.rcParams.update(MID_RC)
# Distribution of categorical attributes
cat_cols = [c for c in stat_cols if c not in [
    'stat_paseo_maritimo', 'stat_tipo_de_usuario_local', 'stat_tipo_de_usuario_turista']]
bool_cols = [c for c in stat_cols if c in [
    'stat_paseo_maritimo', 'stat_tipo_de_usuario_local', 'stat_tipo_de_usuario_turista']]

if cat_cols and not meta_df.empty:
    n = len(cat_cols) + (1 if bool_cols else 0)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1: axes = [axes]

    labels = {
        'stat_grado_de_ocupacion': {0: 'Low', 1: 'Medium', 2: 'High'},
        'stat_proximidad_al_nucleo_urbano': {0: 'Remote', 1: 'Semi', 2: 'Urban'},
        'stat_composicion_de_la_playa': {0: 'Sand', 1: 'Gravel', 2: 'Pebbles', 3: 'Rocks'},
        'stat_condiciones_de_bano': {0: 'Calm', 1: 'Moderate', 2: 'Strong'},
    }

    for i, col in enumerate(cat_cols):
        counts = meta_df[col].dropna().value_counts().sort_index()
        lbl = labels.get(col, {})
        x_labels = [lbl.get(int(k), str(int(k))) for k in counts.index]
        axes[i].bar(x_labels, counts.values, color='#3498db', alpha=0.8, edgecolor='white')
        axes[i].set_title(col.replace('stat_', '').replace('_', ' ').title())
        axes[i].set_ylabel('Beaches')
        for bar, val in zip(axes[i].patches, counts.values):
            axes[i].text(bar.get_x() + bar.get_width()/2, val + 0.1, str(val), ha='center', fontsize=9)

    if bool_cols:
        ax = axes[-1]
        bool_counts = meta_df[bool_cols].notna().sum(axis=0)
        true_counts = meta_df[bool_cols].sum()
        x = range(len(bool_cols))
        ax.bar(x, true_counts, color='#2ecc71', alpha=0.8, label='Yes', edgecolor='white')
        ax.bar(x, bool_counts - true_counts, bottom=true_counts, color='#e74c3c', alpha=0.8, label='No', edgecolor='white')
        ax.set_xticks(x)
        ax.set_xticklabels([c.replace('stat_', '').replace('tipo_de_usuario_', '') for c in bool_cols], fontsize=8)
        ax.set_title('Boolean Features')
        ax.set_ylabel('Beaches')
        ax.legend(fontsize=8)

    plt.suptitle('Beach Attribute Distributions')
    plt.tight_layout()
    plt.show()
else:
    print('No categorical metadata available')

plt.rcParams.update(BASE_RC)


In [ ]:
plt.rcParams.update(MID_RC)
# Mean occupancy vs beach attributes
beach_occ = df_final.groupby('unique_id').agg(
    mean_y=('y', 'mean'), p90_y=('y', lambda x: np.percentile(x, 90)),
    records=('y', 'size')
).reset_index()

if stat_cols and not meta_df.empty:
    occ_meta = beach_occ.merge(meta_df, on='unique_id', how='left')
    check = {
        'stat_grado_de_ocupacion': 'Occupancy Grade',
        'stat_proximidad_al_nucleo_urbano': 'Urban Proximity',
        'stat_composicion_de_la_playa': 'Composition',
        'stat_condiciones_de_bano': 'Bathing Conditions',
    }
    avail = {k: v for k, v in check.items() if k in occ_meta.columns}

    if avail:
        fig, axes = plt.subplots(1, len(avail), figsize=(5 * len(avail), 4))
        if len(avail) == 1: axes = [axes]
        for ax, (col, title) in zip(axes, avail.items()):
            valid = occ_meta[occ_meta[col].notna()]
            if valid.empty:
                ax.set_visible(False)
                continue
            groups = valid.groupby(col)['p90_y'].apply(list)
            bp = ax.boxplot(groups.values, positions=list(range(len(groups))),
                           patch_artist=True, widths=0.6)
            for patch in bp['boxes']:
                patch.set_facecolor('#3498db')
                patch.set_alpha(0.7)
            ax.set_xticklabels([str(int(g)) for g in groups.index], fontsize=9)
            ax.set_title(title)
            ax.set_ylabel('P90 Occupancy')
        plt.suptitle('P90 Occupancy by Beach Attribute')
        plt.tight_layout()
        plt.show()
else:
    print('No metadata to compare')

plt.rcParams.update(BASE_RC)


In [ ]:
plt.rcParams.update(MID_RC)
# Tourist vs Local beaches: different patterns?
t_col = 'stat_tipo_de_usuario_turista' if 'stat_tipo_de_usuario_turista' in df_final.columns else None

if t_col and df_final[t_col].notna().sum() > 0:
    df_final['_btype'] = df_final[t_col].map({1: 'Tourist', 0: 'Local'})
    typed = df_final[df_final['_btype'].notna()]

    if typed['_btype'].nunique() >= 2:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        colors = {'Tourist': '#e74c3c', 'Local': '#3498db'}

        for btype, color in colors.items():
            bt = typed[typed['_btype'] == btype]
            axes[0].plot(bt.groupby('hour')['y'].mean(), 'o-', label=btype, color=color, lw=2)
            axes[1].plot(bt.groupby('month')['y'].mean(), 'o-', label=btype, color=color, lw=2)
        axes[0].set_xlabel('Hour'); axes[0].set_ylabel('Mean Occupancy')
        axes[0].set_title('Hourly Profile'); axes[0].legend()
        axes[1].set_xlabel('Month'); axes[1].set_ylabel('Mean Occupancy')
        axes[1].set_title('Seasonal Pattern'); axes[1].legend()

        uplifts = []
        for beach in typed['unique_id'].unique():
            bdf = typed[typed['unique_id'] == beach]
            wk = bdf[bdf['is_weekend'] == 0]['y'].mean()
            we = bdf[bdf['is_weekend'] == 1]['y'].mean()
            bt = bdf['_btype'].iloc[0]
            if wk > 0:
                uplifts.append({'beach': beach, 'type': bt, 'uplift': 100 * (we / wk - 1)})
        up_df = pd.DataFrame(uplifts)
        for btype, color in colors.items():
            vals = up_df[up_df['type'] == btype]['uplift']
            if len(vals):
                axes[2].hist(vals, bins=10, alpha=0.6, color=color,
                            label=f'{btype} (mean={vals.mean():.1f}%)', edgecolor='white')
        axes[2].axvline(0, color='k', ls='--', lw=0.5)
        axes[2].set_xlabel('Weekend Uplift (%)'); axes[2].set_ylabel('Beaches')
        axes[2].set_title('Weekend Effect'); axes[2].legend()

        plt.suptitle('Tourist vs Local Beaches')
        plt.tight_layout()
        plt.show()
    else:
        print(f'Only one beach type: {typed["_btype"].unique()}')
    df_final.drop(columns=['_btype'], errors='ignore', inplace=True)
else:
    print('No tourist/local classification available')

plt.rcParams.update(BASE_RC)


In [ ]:
plt.rcParams.update(MID_RC)
# Urban vs Semi-Urban vs Remote beaches: different patterns?
u_col = 'stat_proximidad_al_nucleo_urbano' if 'stat_proximidad_al_nucleo_urbano' in df_final.columns else None

if u_col and df_final[u_col].notna().sum() > 0:
    df_final['_uprox'] = df_final[u_col].map({0: 'Remote', 1: 'Semi-Urban', 2: 'Urban'})
    typed = df_final[df_final['_uprox'].notna()]
    colors = {'Remote': '#2ecc71', 'Semi-Urban': '#f39c12', 'Urban': '#e74c3c'}
    present = [k for k in colors if k in typed['_uprox'].values]

    if len(present) >= 2:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))

        for utype in present:
            bt = typed[typed['_uprox'] == utype]
            axes[0].plot(bt.groupby('hour')['y'].mean(), 'o-', label=utype, color=colors[utype], lw=2)
            axes[1].plot(bt.groupby('month')['y'].mean(), 'o-', label=utype, color=colors[utype], lw=2)
        axes[0].set_xlabel('Hour'); axes[0].set_ylabel('Mean Occupancy')
        axes[0].set_title('Hourly Profile'); axes[0].legend()
        axes[1].set_xlabel('Month'); axes[1].set_ylabel('Mean Occupancy')
        axes[1].set_title('Seasonal Pattern'); axes[1].legend()

        uplifts = []
        for beach in typed['unique_id'].unique():
            bdf = typed[typed['unique_id'] == beach]
            wk = bdf[bdf['is_weekend'] == 0]['y'].mean()
            we = bdf[bdf['is_weekend'] == 1]['y'].mean()
            ut = bdf['_uprox'].iloc[0]
            if wk > 0:
                uplifts.append({'beach': beach, 'type': ut, 'uplift': 100 * (we / wk - 1)})
        up_df = pd.DataFrame(uplifts)
        for utype in present:
            vals = up_df[up_df['type'] == utype]['uplift']
            if len(vals):
                axes[2].hist(vals, bins=8, alpha=0.5, color=colors[utype],
                            label=f'{utype} (mean={vals.mean():.1f}%)', edgecolor='white')
        axes[2].axvline(0, color='k', ls='--', lw=0.5)
        axes[2].set_xlabel('Weekend Uplift (%)'); axes[2].set_ylabel('Beaches')
        axes[2].set_title('Weekend Effect'); axes[2].legend()

        plt.suptitle('Urban Proximity: Beach Behavior Patterns')
        plt.tight_layout()
        plt.show()
    else:
        print(f'Only one proximity type: {typed["_uprox"].unique()}')
    df_final.drop(columns=['_uprox'], errors='ignore', inplace=True)
else:
    print('No urban proximity classification available')
plt.rcParams.update(BASE_RC)


In [ ]:
# Static features summary for TFT stat_exog_list
print('Static features for TFT stat_exog_list:')
print('  From data:     stat_mean_y, stat_cv')
print('  From metadata:')
new_stat = [
    ('stat_grado_de_ocupacion',          'LOW=0, MED=1, HIGH=2'),
    ('stat_proximidad_al_nucleo_urbano',  'REMOTE=0, SEMI=1, URBAN=2'),
    ('stat_composicion_de_la_playa',      'SAND=0, GRAVEL=1, PEBBLES=2, ROCKS=3'),
    ('stat_condiciones_de_bano',          'CALM=0, MODERATE=1, STRONG=2'),
    ('stat_paseo_maritimo',               'Promenade (0/1)'),
    ('stat_tipo_de_usuario_turista',      'Tourist beach (0/1)'),
    ('stat_tipo_de_usuario_local',        'Local beach (0/1)'),
]
for name, desc in new_stat:
    present = name in df_final.columns and df_final[name].notna().any()
    tag = '✓' if present else '✗'
    print(f'    {tag} {name:45s} {desc}')


---
# Summary & TFT Recommendations

In [ ]:
n_suspect_c = len(suspect_beaches_cache) if HAS_CACHE else 0
n_suspect_d = len(suspect_beaches_django)
n_shifted = len(shifted) if 'shifted' in dir() else 0

print('=' * 70)
print('DATASET CLEANING SUMMARY')
print('=' * 70)
print(f'Raw records:           {len(raw):,}')
print(f'After hour filter:     {len(df):,}')
print(f'After stuck filter:    {len(df_clean):,}')
print(f'After auto removal:    {len(df_final):,}')
print()
if HAS_CACHE:
    print(f'Cache clean:           {len(cache_clean):,} records, {cache_clean["unique_id"].nunique()} cameras')
    print(f'  Auto-removed:        {n_suspect_c} (seasonality)')
    n_legacy = len(legacy_exclude_cache) if 'legacy_exclude_cache' in dir() else 0
    print(f'  Legacy excluded:     {n_legacy} (folder list)')
    n_manual = len(exclude_c) if 'exclude_c' in dir() else 0
    print(f'  Manual excluded:     {n_manual}')
print(f'Django clean:          {len(django_clean):,} records, {django_clean["unique_id"].nunique()} cameras')
print(f'  Auto-removed:        {n_suspect_d} (seasonality)')
print(f'  Manual excluded:     0 (fully automated)')
if HAS_CACHE and n_shifted > 0:
    print(f'Magnitude shifts:      {n_shifted}')
print()
we_d = django_clean[om_cols].notna().all(axis=1).mean()
print(f'Weather: django={we_d*100:.0f}%')
if HAS_CACHE:
    we_c = cache_clean[om_cols].notna().all(axis=1).mean()
    print(f'         cache={we_c*100:.0f}%')

In [ ]:
MIN_ROWS = 1000

counts = df_final.groupby('unique_id').size()
small = counts[counts < MIN_ROWS]
keep = counts[counts >= MIN_ROWS].index

if len(small):
    print(f'Removing {len(small)} cameras with < {MIN_ROWS} rows:')
    for cam, n in small.sort_values().items():
        print(f'  {cam:45s} {n:>5,} rows')

before = len(df_final)
df_final = df_final[df_final['unique_id'].isin(keep)].reset_index(drop=True)
if HAS_CACHE:
    cache_clean = cache_clean[cache_clean['unique_id'].isin(keep)].reset_index(drop=True)
django_clean = django_clean[django_clean['unique_id'].isin(keep)].reset_index(drop=True)

print(f'\n{before:,} → {len(df_final):,} records ({df_final["unique_id"].nunique()} cameras kept)')

In [ ]:
import os

os.makedirs(OUT_DIR, exist_ok=True)

django_clean['period'] = 'django_2025'
django_clean.to_csv(f'{OUT_DIR}/django_2025_clean.csv', index=False)

if HAS_CACHE:
    cache_clean['period'] = 'cache_2022'
    cache_clean.to_csv(f'{OUT_DIR}/cache_2022_clean.csv', index=False)
    df_final = pd.concat([cache_clean, django_clean], ignore_index=True)
else:
    df_final = django_clean.copy()
df_final.to_csv(f'{OUT_DIR}/all_clean.csv', index=False)

datasets = [('django_2025', django_clean)]
if HAS_CACHE:
    datasets.insert(0, ('cache_2022', cache_clean))
datasets.append(('all', df_final))

for name, subset in datasets:
    print(f'{name:15s} | {len(subset):>7,} records | {subset["unique_id"].nunique():>2} beaches | '
          f'{subset["ds"].min().date()} → {subset["ds"].max().date()}')
print(f'\nSaved to {OUT_DIR}/')

In [ ]:
print(f'{"="*70}')
print(f'DATASET SUMMARY')
print(f'{"="*70}')
print(f'Records:    {len(df_final):,}')
print(f'Beaches:    {df_final["unique_id"].nunique()}')
print(f'Date range: {df_final["ds"].min().date()} → {df_final["ds"].max().date()}')
print(f'Hours:      {df_final["hour"].min()}h – {df_final["hour"].max()}h')
print(f'Sources:    {df_final["source"].value_counts().to_dict()}')

print(f'\nColumn groups:')
col_groups = {
    'Target':    ['y'],
    'Temporal':  [c for c in ['hour','day_of_week','month','day_of_year','is_weekend','is_holiday'] if c in df_final.columns],
    'Weather':   [c for c in df_final.columns if c.startswith('om_')],
    'Static':    [c for c in df_final.columns if c.startswith('stat_')],
    'Identity':  [c for c in ['unique_id','slug','webcam_id','source','period'] if c in df_final.columns],
}
for group, cols in col_groups.items():
    if cols:
        cov = df_final[cols].notna().mean().mean() * 100
        print(f'  {group:12s} {len(cols):>3} cols  coverage: {cov:.0f}%')

print(f'\nPer-beach summary:')
summary = df_final.groupby('unique_id').agg(
    records=('y', 'size'), mean_y=('y', 'mean'),
    p90_y=('y', lambda x: np.percentile(x, 90)),
    start=('ds', 'min'), end=('ds', 'max'), source=('source', 'first')
).sort_values('records', ascending=False)
for uid, r in summary.iterrows():
    print(f'  {uid:40s} {r["records"]:>6,} rows  mean={r["mean_y"]:.0f}  P90={r["p90_y"]:.0f}  '
          f'{r["start"].date()}→{r["end"].date()}  [{r["source"]}]')